# Procedural CCLE protein prediction partitions — unified notebook

This notebook consolidates the three previously merged CCLE notebooks into one pipeline. Shared data loading, H5 split recovery, ENet/XGB loading, cascade logic, scTranslator evaluation, plotting, and export logic are executed only once.

The only intentional branch is the **same-gene one-feature proxy definition**. All three original proxy variants are preserved and evaluated from the same train/test split:

1. **Raw RNA proxy** — `y_pred = x_test`.
2. **Train z-score mapping** — gene values are standardized with training RNA statistics and mapped to the training protein mean/std.
3. **Train linear calibration** — `LinearRegression` is fitted on training RNA/protein pairs only.

The notebook computes the proxy variants together, then runs the same Pearson, cosine-similarity, and MSE cascades for each mode without reloading data or redefining helpers.

## 1. Setup and project paths

All imports and user-adjustable settings live here. The H5 split is recovered from the saved models, so the old unused random-split settings were removed.

In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "4_ccle_cascade"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
import math
import importlib.util as importlib_util
import re
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from IPython.display import display
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"

X_PATH = DATA_DIR / "transcriptomics_filtered.joblib"
Y_PATH = DATA_DIR / "proteomics_matched.joblib"
PEARSON_PATH = DATA_DIR / "full_pearson100.joblib"

MODEL_DIRS = {
    "xgb": ROOT / "data" / "results_xgb",
    "xgb_zbig": ROOT / "data" / "res_xgb_zbig",
    "enet": ROOT / "data" / "results_enet",
    "enet_zbig": ROOT / "data" / "res_enet_zbig",
}

SCTRANSLATOR_BUNDLE_PATHS = {
    "2M": ROOT / "data" / "sc_predictions_2M.npz",
    "160K": ROOT / "data" / "sc_predictions_160K.npz",
    "10K": ROOT / "data" / "sc_predictions_10K.npz",
}

# ---------------------------------------------------------------------
# Analysis settings
# ---------------------------------------------------------------------
PEARSON_GOOD_THRESHOLDS = [0.30, 0.50, 0.70]
COSINE_GOOD_THRESHOLDS = [0.30, 0.50, 0.70]
MSE_GOOD_THRESHOLDS = [0.30, 0.50, 0.70]

PEARSON_SWEEP_THRESHOLDS = [round(x, 2) for x in np.arange(0.10, 0.85, 0.05)]
COSINE_SWEEP_THRESHOLDS = [round(x, 2) for x in np.arange(0.10, 0.85, 0.05)]
MSE_SWEEP_THRESHOLDS = [round(x, 2) for x in np.arange(0.10, 1.05, 0.05)]

PROXY_MODES_TO_RUN = ["raw", "train_zscore", "train_linear"]

PROXY_MODE_CONFIG = {
    "raw": {
        "label": "Raw RNA proxy",
        "run_label": "proxy_one_gene",
        "normalization": "raw_gene_value",
    },
    "train_zscore": {
        "label": "Train z-score proxy",
        "run_label": "proxy_one_gene_train_zscore",
        "normalization": "train_zscore_gene_to_protein_scale",
    },
    "train_linear": {
        "label": "Train linear-regression proxy",
        "run_label": "proxy_one_gene_train_linear",
        "normalization": "train_linear_regression",
    },
}

FIGURE_ROOT = FIG_DIR
EXPORT_ROOT = ROOT / "procedural_unified_tables"
SAVE_FIGURES = True
SAVE_OUTPUTS = False
SAVE_CASCADE_PROTEIN_EXPORTS = True

# Publication typography: every explicit plotting font is multiplied by 2.
PLOT_FONT_SCALE = 2.0
# The canvas grows less than the font so text is visibly larger while remaining unclipped.
PLOT_CANVAS_SCALE = 1.55

plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 26,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
    "lines.linewidth": 2.0,
    "patch.linewidth": 1.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.format": "pdf",
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

print("Root:", ROOT)
print("Data dir:", DATA_DIR)
print("PDF figure dir:", FIGURE_ROOT)
print("Proxy modes:", PROXY_MODES_TO_RUN)
print("Model folders:")
for name, folder in MODEL_DIRS.items():
    print(f"  {name:10s}: {folder}  exists={folder.exists()}")
print("\nscTranslator bundles:")
for name, path in SCTRANSLATOR_BUNDLE_PATHS.items():
    print(f"  {name:5s}: {path}  exists={path.exists()}")


## 2. Load shared CCLE utilities once

The notebook still uses the existing `utils_ccle.py` for shared H5 loading, metrics, plotting, and scTranslator helpers. It is imported once and never overridden by mode-specific proxy functions.

In [ ]:
utils_candidates = []
for candidate in [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "notebooks",
    Path.cwd().parent / "notebooks",
    Path.cwd() / "notebooks2",
    Path.cwd().parent / "notebooks2",
]:
    utils_candidates.append(candidate / "utils_ccle.py")
    utils_candidates.append(candidate / "src" / "utils_ccle.py")

utils_path = next((p for p in utils_candidates if p.exists()), None)
if utils_path is None:
    raise FileNotFoundError(
        "Could not find utils_ccle.py. Place it next to this notebook or at src/utils_ccle.py."
    )

spec = importlib_util.spec_from_file_location("utils_ccle", utils_path)
uc = importlib_util.module_from_spec(spec)
sys.modules["utils_ccle"] = uc
spec.loader.exec_module(uc)

for name in getattr(uc, "__all__", []):
    globals()[name] = getattr(uc, name)

uc.set_module_globals(globals())
print("Loaded utilities:", uc.__file__)

## 3. Load transcriptomics, proteomics, and Pearson-correlation data

Proteomics is converted to samples × proteins if needed. The old Spearman matrix load was removed because none of the merged analyses used it downstream.

In [ ]:
for path in [X_PATH, Y_PATH, PEARSON_PATH]:
    require_file(path)

X_raw = joblib.load(X_PATH)
Y_raw = joblib.load(Y_PATH)
pearson_corr_obj = joblib.load(PEARSON_PATH)

X = X_raw.copy()
Y0 = Y_raw.copy()

X.index = X.index.map(str)
X.columns = X.columns.map(str)
Y0.index = Y0.index.map(str)
Y0.columns = Y0.columns.map(str)

# Detect proteomics orientation and convert to rows=samples, columns=proteins.
x_samples = set(X.index)
y_index_overlap = len(x_samples & set(Y0.index))
y_column_overlap = len(x_samples & set(Y0.columns))

if y_index_overlap >= y_column_overlap:
    Y = Y0.copy()
    proteomics_orientation = "samples_as_rows__proteins_as_columns"
else:
    Y = Y0.T.copy()
    proteomics_orientation = "proteins_as_rows__samples_as_columns__transposed"

Y.index = Y.index.map(str)
Y.columns = Y.columns.map(str)

common_samples = X.index.intersection(Y.index)
X = X.loc[common_samples].copy()
Y = Y.loc[common_samples].copy()

GT_scaled = Y.copy()
protein_universe = pd.Index(Y.columns.astype(str)).unique()

print("Loaded data")
print("- Transcriptomics shape:", X.shape, "(samples x genes)")
print("- Proteomics shape:     ", Y.shape, "(samples x proteins)")
print("- Proteomics orientation:", proteomics_orientation)
print("- Common samples:", len(common_samples))
print("- Proteins in universe:", len(protein_universe))
print("- Pearson object type:", type(pearson_corr_obj), "shape:", getattr(pearson_corr_obj, "shape", None))

uc.set_module_globals(globals())

## 4. Load saved ENet/XGB summaries

Only saved metrics and metadata are read from `.h5` files; models are not retrained.

In [ ]:
model_rows = []
folder_counts = {}

for label, folder in MODEL_DIRS.items():
    folder = Path(folder)
    files = sorted(folder.glob("*.h5")) if folder.exists() else []
    folder_counts[label] = len(files)

    for path in files:
        row = load_h5_summary(path, label)
        if "h5_path" not in row:
            row["h5_path"] = str(path)
        model_rows.append(row)

model_results = pd.DataFrame(model_rows)

if len(model_results):
    for c in [
        "test_mse",
        "test_pearson_corr",
        "test_cosine_similarity",
        "selected_feature_count",
        "train_n",
        "test_n",
        "select_k",
        "file_size_mb",
    ]:
        if c in model_results.columns:
            model_results[c] = pd.to_numeric(model_results[c], errors="coerce")

    model_results = add_family_group(model_results)

print("Saved model file counts")
for label, n in folder_counts.items():
    print(f"- {label:12s}: {n:6d} H5 files")

print("\nSaved model result table")
print("- Saved model rows:", len(model_results))
print("- Unique proteins: ", model_results["protein"].nunique() if len(model_results) and "protein" in model_results.columns else 0)

if len(model_results) and "load_error" in model_results.columns:
    n_errors = int(model_results["load_error"].notna().sum())
    print("- H5 load errors:  ", n_errors)
    if n_errors:
        display(model_results.loc[model_results["load_error"].notna(), ["h5_path", "load_error"]].head(10))

if len(model_results):
    display(
        model_results.groupby(["family_group", "run_label"])["protein"]
        .nunique()
        .sort_values(ascending=False)
        .to_frame("proteins")
    )
    display(model_results.head())
else:
    raise RuntimeError("No saved ENet/XGB .h5 result files were found in MODEL_DIRS.")

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass

## 5. Recover the exact H5 test split

Every proxy mode and scTranslator is evaluated on the same test samples stored by the ENet/XGB results.

In [ ]:
if "h5_path" not in model_results.columns:
    raise RuntimeError("model_results has no h5_path column, so the H5 test split cannot be read.")

split_rows = []
split_samples_by_hash = {}

h5_paths = (
    model_results["h5_path"]
    .dropna()
    .astype(str)
    .loc[lambda s: s.str.len() > 0]
    .unique()
)

for path in h5_paths:
    try:
        h5_test_samples = read_test_samples_from_h5(path)
        if h5_test_samples is None:
            split_rows.append({"h5_path": path, "readable": False, "test_n": np.nan, "split_hash": None})
            continue

        h5_test_samples = pd.Index(h5_test_samples.astype(str))
        h = split_hash(h5_test_samples)
        split_samples_by_hash[h] = h5_test_samples
        split_rows.append({"h5_path": path, "readable": True, "test_n": len(h5_test_samples), "split_hash": h})

    except Exception as e:
        split_rows.append({
            "h5_path": path,
            "readable": False,
            "test_n": np.nan,
            "split_hash": None,
            "error": f"{type(e).__name__}: {e}",
        })

h5_split_check = pd.DataFrame(split_rows)

print("H5 test split readability")
display(
    h5_split_check
    .groupby(["readable", "split_hash"], dropna=False)
    .agg(files=("h5_path", "count"), test_n=("test_n", "median"))
    .reset_index()
    .sort_values(["readable", "files"], ascending=[False, False])
)

readable = h5_split_check[h5_split_check["readable"]].copy()
if readable.empty:
    raise RuntimeError(
        "Could not read test_samples from any H5 file. "
        "The saved XGB/ENet files probably do not store the test split."
    )

unique_hashes = readable["split_hash"].dropna().unique()
if len(unique_hashes) != 1:
    display(
        readable
        .groupby("split_hash")
        .agg(files=("h5_path", "count"), test_n=("test_n", "median"))
        .sort_values("files", ascending=False)
    )
    raise RuntimeError(
        f"Found {len(unique_hashes)} different H5 test splits. "
        "A single proxy/scTranslator split cannot fairly match all models."
    )

model_split_hash = unique_hashes[0]
model_test_samples = pd.Index(split_samples_by_hash[model_split_hash].astype(str))
common_samples_index = pd.Index(np.asarray(common_samples, dtype=str))

test_samples = model_test_samples.intersection(common_samples_index)
train_samples = pd.Index([s for s in common_samples_index if s not in set(test_samples)])

print("\nUsing exact XGB/ENet H5 test split")
print("- Split hash:", model_split_hash)
print("- Test samples:", len(test_samples))
print("- Train samples:", len(train_samples))
print("- Test fraction:", len(test_samples) / len(common_samples_index))
print("- First test samples:", list(test_samples[:5]))

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass

## 6. Define the three proxy transformations once

The helper functions below are mode-agnostic and are defined only here. Proxy eligibility is based on **training-sample same-gene Pearson correlation**; the precomputed full-data correlation is retained only as reporting metadata.

In [ ]:
def zscore_gene_to_protein_scale(x_train, y_train, x_test):
    """Map test RNA z-scores onto the training protein mean/std."""
    x_train = np.asarray(x_train, dtype=float).reshape(-1)
    y_train = np.asarray(y_train, dtype=float).reshape(-1)
    x_test = np.asarray(x_test, dtype=float).reshape(-1)

    finite_x_train = np.isfinite(x_train)
    finite_y_train = np.isfinite(y_train)
    y_pred = np.full(x_test.shape, np.nan, dtype=float)

    if finite_x_train.sum() == 0 or finite_y_train.sum() == 0:
        return y_pred

    x_mean = float(np.mean(x_train[finite_x_train]))
    x_std = float(np.std(x_train[finite_x_train], ddof=0))
    y_mean = float(np.mean(y_train[finite_y_train]))
    y_std = float(np.std(y_train[finite_y_train], ddof=0))

    test_mask = np.isfinite(x_test)
    if not test_mask.any():
        return y_pred

    if not np.isfinite(x_std) or x_std <= 0:
        y_pred[test_mask] = y_mean
        return y_pred

    if not np.isfinite(y_std):
        y_std = 0.0

    gene_z = (x_test[test_mask] - x_mean) / x_std
    y_pred[test_mask] = y_mean + y_std * gene_z
    return y_pred


def linear_scale_prediction(x_train, y_train, x_test):
    """Fit train-only one-gene LinearRegression and predict finite test RNA values."""
    x_train = np.asarray(x_train, dtype=float).reshape(-1)
    y_train = np.asarray(y_train, dtype=float).reshape(-1)
    x_test = np.asarray(x_test, dtype=float).reshape(-1)

    train_mask = np.isfinite(x_train) & np.isfinite(y_train)
    y_pred = np.full(x_test.shape, np.nan, dtype=float)
    if train_mask.sum() == 0:
        return y_pred

    reg = LinearRegression()
    reg.fit(x_train[train_mask].reshape(-1, 1), y_train[train_mask])

    test_mask = np.isfinite(x_test)
    if test_mask.any():
        y_pred[test_mask] = reg.predict(x_test[test_mask].reshape(-1, 1))
    return y_pred


def proxy_prediction(mode, x_train, y_train, x_test):
    if mode == "raw":
        return np.asarray(x_test, dtype=float).reshape(-1)
    if mode == "train_zscore":
        return zscore_gene_to_protein_scale(x_train, y_train, x_test)
    if mode == "train_linear":
        return linear_scale_prediction(x_train, y_train, x_test)
    raise ValueError(f"Unknown proxy mode: {mode}")

## 7. Build all same-gene proxy variants in one pass

The expensive eligibility calculation is performed once. Each eligible protein is then scored under the three proxy transformations on the fixed H5 test split.

In [ ]:
possible_same_name = sorted(set(Y.columns.astype(str)).intersection(set(X.columns.astype(str))))

print("Precomputing full-data same-name Pearson lookup for reporting metadata...")
pearson_lookup_full = build_same_name_corr_lookup_fast(pearson_corr_obj, possible_same_name)

print("Computing train-only same-gene Pearson eligibility once...")
eligibility_lookup = compute_same_gene_corr_lookup(
    possible_same_name,
    train_samples,
    method="pearson",
    verbose=True,
)

proxy_rows = []
for protein in possible_same_name:
    corr_train = eligibility_lookup.get(protein, np.nan)
    if not np.isfinite(corr_train):
        continue

    corr_full = pearson_lookup_full.get(protein, np.nan)
    x_train = X.loc[train_samples, protein].to_numpy(dtype=float)
    y_train = Y.loc[train_samples, protein].to_numpy(dtype=float)
    x_test = X.loc[test_samples, protein].to_numpy(dtype=float)
    y_true = Y.loc[test_samples, protein].to_numpy(dtype=float)

    for mode in PROXY_MODES_TO_RUN:
        cfg = PROXY_MODE_CONFIG[mode]
        y_pred = proxy_prediction(mode, x_train, y_train, x_test)
        metrics = compute_metrics(y_true, y_pred)

        if mode == "raw":
            baseline_meta = {
                "baseline": "same_name_gene_raw_proxy",
                "test_split_source": "read_from_xgb_enet_h5",
            }
        elif mode == "train_zscore":
            baseline_meta = {
                "baseline": "same_name_gene_train_zscore_proxy",
                "normalization": "gene_zscore_to_train_protein_scale",
                "calibration_fit_samples": "train_only",
                "test_split_source": "read_from_xgb_enet_h5",
            }
        else:
            baseline_meta = {
                "baseline": "same_name_gene_linear_regression_proxy",
                "calibration_model": "sklearn.linear_model.LinearRegression",
                "calibration_fit_samples": "train_only",
                "test_split_source": "read_from_xgb_enet_h5",
            }

        proxy_rows.append({
            "protein": protein,
            "run_label": cfg["run_label"],
            "proxy_mode": mode,
            "proxy_mode_label": cfg["label"],
            "model_family": "same_gene_proxy",
            "family_group": "proxy",
            "source_folder": "proxy_one_gene",
            "normalization": cfg["normalization"],
            "h5_path": "",
            "proxy_type": "one_gene",
            "same_gene_corr_train": float(corr_train),
            "corr_scope": "train_only",
            "corr_method": "pearson",
            "same_gene_corr_full_reference_only": float(corr_full) if np.isfinite(corr_full) else np.nan,
            "test_mse": metrics["test_mse"],
            "test_pearson_corr": metrics["test_pearson_corr"],
            "test_cosine_similarity": metrics["test_cosine_similarity"],
            "n_test_used": metrics.get("n_test_used", np.nan),
            "selected_feature_count": 1,
            "train_n": len(train_samples),
            "test_n": len(test_samples),
            "split_hash": model_split_hash,
            "best_params": baseline_meta,
        })

proxy_results_all = pd.DataFrame(proxy_rows)
proxy_results_by_mode = {
    mode: add_family_group(proxy_results_all.loc[proxy_results_all["proxy_mode"] == mode].copy())
    for mode in PROXY_MODES_TO_RUN
}

all_results_by_mode = {
    mode: add_family_group(
        pd.concat([model_results, proxy_results_by_mode[mode]], ignore_index=True, sort=False)
    )
    for mode in PROXY_MODES_TO_RUN
}

print("Same-name gene/protein pairs possible:", len(possible_same_name))
print("Eligible train-only proxy proteins:", proxy_results_all["protein"].nunique())
print("Proxy rows across all modes:", len(proxy_results_all))
display(
    proxy_results_all.groupby(["proxy_mode", "run_label"])
    .agg(
        proteins=("protein", "nunique"),
        pearson_median=("test_pearson_corr", "median"),
        cosine_median=("test_cosine_similarity", "median"),
        mse_median=("test_mse", "median"),
    )
)

## 8. Generic cascade helpers

Pearson, cosine similarity, and MSE now share one cascade implementation instead of three repeated blocks. ENet/XGB best-per-protein tables are computed once because they do not depend on proxy mode.

In [ ]:
# =============================================================================
# CASCADE PLOTTING HELPERS
#
# FINAL scTRANSLATOR DESIGN:
#
# All proteins -> Proxy -> ElasticNet -> XGBoost
#                                      |--> scTranslator 2M   --\
#                                      |--> scTranslator 160K ---+--> Unresolved
#                                      \--> scTranslator 10K  --/
#
# Changes:
# - scTranslator 2M / 160K / 10K are three DISTINCT parallel boxes
# - Unresolved is a SEPARATE red box farther to the right
# - XGBoost connects ONLY to the three scTranslator boxes
# - the three scTranslator paths visually converge
# - ONE final arrow then leads to Unresolved
# - scTranslator boxes are narrower
# - Unresolved box is narrower
# - no intermediate "Best size" block
# - no "% of remaining" inside scTranslator boxes
# - original stacked design preserved with larger legend / spacing
# =============================================================================

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from matplotlib.patches import Patch, FancyBboxPatch
from matplotlib.ticker import MaxNLocator


CASCADE_ORDER_BASE = [
    "Proxy one gene",
    "ENet",
    "XGB",
    "Still not good",
]

CASCADE_ORDER_WITH_SC = [
    "Proxy one gene",
    "ENet",
    "XGB",
    "scTranslator 2M",
    "scTranslator 160K",
    "scTranslator 10K",
    "Still not good after scTranslator",
]

SC_RESCUE_ORDER = [
    "scTranslator 2M",
    "scTranslator 160K",
    "scTranslator 10K",
    "Still not good after scTranslator",
]


COLOR_MAP = {
    "Proxy one gene": "#4C78A8",
    "ENet": "#54A24B",
    "XGB": "#F58518",
    "scTranslator 2M": "#B279A2",
    "scTranslator 160K": "#9D755D",
    "scTranslator 10K": "#BAB0AC",
    "Still not good": "#E45756",
    "Still not good after scTranslator": "#E45756",
}


METRIC_CONFIG = {
    "pearson": {
        "column": "test_pearson_corr",
        "higher": True,
        "main_thresholds": PEARSON_GOOD_THRESHOLDS,
        "sweep_thresholds": PEARSON_SWEEP_THRESHOLDS,
        "title": "Pearson correlation",
        "row_label": lambda thr: f"Pearson >= {thr:.2f}",
    },
    "cosine": {
        "column": "test_cosine_similarity",
        "higher": True,
        "main_thresholds": COSINE_GOOD_THRESHOLDS,
        "sweep_thresholds": COSINE_SWEEP_THRESHOLDS,
        "title": "Cosine similarity",
        "row_label": lambda thr: f"Cosine >= {thr:.2f}",
    },
    "mse": {
        "column": "test_mse",
        "higher": False,
        "main_thresholds": MSE_GOOD_THRESHOLDS,
        "sweep_thresholds": MSE_SWEEP_THRESHOLDS,
        "title": "MSE",
        "row_label": lambda thr: f"MSE <= {thr:.2f}",
    },
}


CASCADE_LEGEND_LABELS = {
    "Proxy one gene": "Same-gene proxy — reaches the threshold first",
    "ENet": "ElasticNet — among proteins remaining after the proxy",
    "XGB": "XGBoost — among proteins remaining after proxy + ElasticNet",
    "scTranslator 2M": "scTranslator 2M — among remaining base-model failures",
    "scTranslator 160K": "scTranslator 160K — among remaining base-model failures",
    "scTranslator 10K": "scTranslator 10K — among remaining base-model failures",
    "Still not good": "Unresolved — no base model reaches the threshold",
    "Still not good after scTranslator": "Unresolved — no tested model reaches the threshold",
}


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def _contrasting_text_color(color):
    red, green, blue = mcolors.to_rgb(color)

    luminance = (
        0.2126 * red
        + 0.7152 * green
        + 0.0722 * blue
    )

    return "#111111" if luminance > 0.68 else "white"


def _cascade_summary_pivot(summary, order):
    required = {
        "threshold",
        "stage",
        "proteins",
    }

    missing = required.difference(summary.columns)

    if missing:
        raise ValueError(
            "Cascade summary is missing required columns: "
            f"{sorted(missing)}"
        )

    work = summary.copy()

    work["threshold"] = pd.to_numeric(
        work["threshold"],
        errors="coerce",
    )

    work["proteins"] = pd.to_numeric(
        work["proteins"],
        errors="coerce",
    ).fillna(0)

    work = work.dropna(
        subset=["threshold"]
    )

    thresholds = sorted(
        work["threshold"].unique()
    )

    if not thresholds:
        raise ValueError(
            "Cascade summary contains no numeric thresholds."
        )

    return (
        work.pivot_table(
            index="threshold",
            columns="stage",
            values="proteins",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(
            index=thresholds,
            columns=order,
            fill_value=0,
        )
        .astype(float)
    )


def _format_flow_percent(value):
    value = float(value)

    if 0 < abs(value) < 0.1:
        return f"{value:.2f}%"

    return f"{value:.1f}%"


# =============================================================================
# ORIGINAL DESIGN — STACKED BAR
# =============================================================================

def plot_stacked_partition_rows(
    summary,
    *,
    order,
    save_name=None,
    figsize=None,
    row_label_func=None,
):
    required = {
        "threshold",
        "stage",
        "proteins",
    }

    missing = required.difference(summary.columns)

    if missing:
        raise ValueError(
            "Cascade summary is missing required columns: "
            f"{sorted(missing)}"
        )

    work = summary.copy()

    work["threshold"] = pd.to_numeric(
        work["threshold"],
        errors="coerce",
    )

    work["proteins"] = pd.to_numeric(
        work["proteins"],
        errors="coerce",
    ).fillna(0)

    work = work.dropna(
        subset=["threshold"]
    )

    thresholds = sorted(
        work["threshold"].unique()
    )

    if not thresholds:
        raise ValueError(
            "Cascade summary contains no numeric thresholds."
        )

    pivot = (
        work.pivot_table(
            index="threshold",
            columns="stage",
            values="proteins",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(
            index=thresholds,
            columns=order,
            fill_value=0,
        )
        .astype(float)
    )

    n_rows = len(pivot)

    if figsize is None:
        width = 14.8
        height = max(
            7.0,
            1.20 * n_rows + 3.35,
        )
    else:
        width = max(
            float(figsize[0]),
            14.3,
        )

        height = max(
            float(figsize[1]) * 1.45,
            0.90 * n_rows + 3.8,
        )

    fig, ax = plt.subplots(
        figsize=(width, height)
    )
    fig.set_size_inches(
        np.asarray(fig.get_size_inches()) * PLOT_CANVAS_SCALE,
        forward=True,
    )

    y_positions = (
        np.arange(
            n_rows,
            dtype=float,
        )
        * 1.35
    )

    left = np.zeros(
        n_rows,
        dtype=float,
    )

    totals = (
        pivot.sum(axis=1)
        .to_numpy(dtype=float)
    )

    max_total = (
        float(np.nanmax(totals))
        if len(totals)
        else 1.0
    )

    max_total = max(
        max_total,
        1.0,
    )

    bar_height = 0.78

    for stage_index, stage in enumerate(order):

        values = pivot[
            stage
        ].to_numpy(dtype=float)

        color = COLOR_MAP.get(
            stage,
            "#8C8C8C",
        )

        bars = ax.barh(
            y_positions,
            values,
            left=left,
            height=bar_height,
            color=color,
            edgecolor="white",
            linewidth=1.15,
            label=stage,
            zorder=3,
        )

        for row_index, (
            bar,
            value,
            total,
        ) in enumerate(
            zip(
                bars,
                values,
                totals,
            )
        ):

            if value <= 0 or total <= 0:
                continue

            percent = (
                100.0
                * value
                / total
            )

            centre = (
                left[row_index]
                + 0.5 * value
            )

            if percent >= 4.5:

                ax.text(
                    centre,
                    y_positions[row_index],
                    (
                        f"{int(round(value)):,}\n"
                        f"{percent:.1f}%"
                    ),
                    ha="center",
                    va="center",
                    fontsize=11.7 * PLOT_FONT_SCALE,
                    fontweight="bold",
                    color=_contrasting_text_color(
                        color
                    ),
                    zorder=5,
                )

            else:

                vertical_offset = (
                    -0.54
                    if stage_index % 2 == 0
                    else 0.54
                )

                ax.annotate(
                    f"{int(round(value)):,}",
                    xy=(
                        centre,
                        y_positions[row_index],
                    ),
                    xytext=(
                        centre,
                        y_positions[row_index]
                        + vertical_offset,
                    ),
                    ha="center",
                    va="center",
                    fontsize=10.2 * PLOT_FONT_SCALE,
                    fontweight="bold",
                    color="#2F2F2F",
                    arrowprops={
                        "arrowstyle": "-",
                        "color": color,
                        "linewidth": 1.0,
                        "shrinkA": 1,
                        "shrinkB": 1,
                    },
                    zorder=6,
                )

        left += values

    labels = [
        (
            row_label_func(threshold)
            if row_label_func is not None
            else f"{threshold:g}"
        )
        for threshold in thresholds
    ]

    ax.set_yticks(
        y_positions
    )

    ax.set_yticklabels(
        labels,
        fontsize=12.5 * PLOT_FONT_SCALE,
    )

    ax.invert_yaxis()

    ax.set_xlabel(
        "Number of proteins",
        fontsize=13.5 * PLOT_FONT_SCALE,
        labelpad=12,
    )

    ax.set_xlim(
        0,
        max_total,
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(
            integer=True,
            nbins=7,
        )
    )

    ax.tick_params(
        axis="x",
        labelsize=11.5,
    )

    ax.grid(
        axis="x",
        alpha=0.20,
        linewidth=0.8,
        zorder=0,
    )

    ax.grid(
        axis="y",
        visible=False,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    ax.tick_params(
        axis="y",
        length=0,
        pad=8,
    )

    present_stages = [
        stage
        for stage in order
        if float(
            pivot[stage].sum()
        ) > 0
    ]

    handles = [
        Patch(
            facecolor=COLOR_MAP.get(
                stage,
                "#8C8C8C",
            ),
            edgecolor="white",
            label=CASCADE_LEGEND_LABELS.get(
                stage,
                stage,
            ),
        )
        for stage in present_stages
    ]

    legend_columns = (
        2
        if len(handles) > 4
        else min(
            2,
            max(
                1,
                len(handles),
            ),
        )
    )

    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(
            0.5,
            0.025,
        ),
        ncol=legend_columns,
        frameon=False,
        fontsize=12.0 * PLOT_FONT_SCALE,
        handlelength=1.9,
        handleheight=1.0,
        columnspacing=2.1,
        labelspacing=0.90,
    )

    legend_rows = max(
        1,
        math.ceil(
            len(handles)
            / max(
                legend_columns,
                1,
            )
        ),
    )

    bottom_margin = min(
        0.52,
        0.24
        + 0.085
        * legend_rows,
    )

    fig.subplots_adjust(
        left=0.205,
        right=0.99,
        top=0.975,
        bottom=bottom_margin,
    )

    if save_name and SAVE_FIGURES:

        output_path = (
            Path(FIGURE_DIR)
            / Path(str(save_name)).with_suffix(".pdf")
        )

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig.savefig(
            output_path,
            format="pdf",
            bbox_inches="tight",
            pad_inches=0.50,
        )

    plt.show()

    return fig, ax


# =============================================================================
# FLOW BOX / ARROW / CONNECTOR HELPERS
# =============================================================================

def _flow_box(
    ax,
    x,
    y,
    width,
    height,
    text,
    *,
    facecolor,
    edgecolor="white",
    fontsize=15.0 * PLOT_FONT_SCALE,
    textcolor=None,
):
    patch = FancyBboxPatch(
        (
            x - width / 2.0,
            y - height / 2.0,
        ),
        width,
        height,
        boxstyle=(
            "round,pad=0.014,"
            "rounding_size=0.016"
        ),
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=1.60,
        zorder=3,
    )

    ax.add_patch(
        patch
    )

    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold",
        color=(
            textcolor
            or _contrasting_text_color(
                facecolor
            )
        ),
        zorder=4,
        linespacing=1.12,
    )

    return patch


def _flow_arrow(
    ax,
    x1,
    y1,
    x2,
    y2,
    *,
    label=None,
    label_y_offset=0.075,
):
    ax.annotate(
        "",
        xy=(
            x2,
            y2,
        ),
        xytext=(
            x1,
            y1,
        ),
        arrowprops={
            "arrowstyle": "-|>",
            "linewidth": 2.0,
            "color": "#666666",
            "shrinkA": 3,
            "shrinkB": 3,
            "mutation_scale": 16,
        },
        zorder=2,
    )

    if label:

        ax.text(
            (
                x1 + x2
            )
            / 2.0,
            (
                y1 + y2
            )
            / 2.0
            + label_y_offset,
            label,
            ha="center",
            va="bottom",
            fontsize=13.7 * PLOT_FONT_SCALE,
            fontweight="medium",
            color="#555555",
            zorder=5,
        )


def _flow_connector(
    ax,
    x1,
    y1,
    x2,
    y2,
    *,
    linewidth=1.9,
):
    """
    Plain connector without an arrowhead.
    Useful for joining the three scTranslator branches
    before the final unresolved arrow.
    """

    ax.plot(
        [x1, x2],
        [y1, y2],
        color="#666666",
        linewidth=linewidth,
        solid_capstyle="round",
        zorder=2,
    )


# =============================================================================
# ALTERNATIVE BASE CASCADE
# =============================================================================

def plot_cascade_flow_alternative(
    summary,
    *,
    order,
    save_name=None,
    row_label_func=None,
    figsize=None,
):
    pivot = _cascade_summary_pivot(
        summary,
        order,
    )

    unresolved = [
        stage
        for stage in order
        if str(stage).startswith(
            "Still not good"
        )
    ]

    if len(unresolved) != 1:
        raise ValueError(
            "Flow artwork expects exactly one unresolved stage."
        )

    unresolved_stage = unresolved[0]

    model_stages = [
        stage
        for stage in order
        if stage != unresolved_stage
    ]

    thresholds = list(
        pivot.index
    )

    n_rows = len(
        thresholds
    )

    if figsize is None:

        figsize = (
            20.8,
            max(
                7.6,
                2.75 * n_rows,
            ),
        )

    fig, axes = plt.subplots(
        n_rows,
        1,
        figsize=figsize,
        squeeze=False,
    )
    fig.set_size_inches(
        np.asarray(fig.get_size_inches()) * PLOT_CANVAS_SCALE,
        forward=True,
    )

    axes = axes[:, 0]

    positions = np.linspace(
        0.245,
        0.935,
        len(model_stages) + 2,
    )

    box_w = min(
        0.140,
        0.62
        / max(
            len(model_stages) + 1,
            1,
        ),
    )

    box_h = 0.43

    headers = (
        ["All proteins"]
        + [
            (
                "Same-gene\nproxy"
                if stage == "Proxy one gene"
                else (
                    "ElasticNet"
                    if stage == "ENet"
                    else (
                        "XGBoost"
                        if stage == "XGB"
                        else stage
                    )
                )
            )
            for stage in model_stages
        ]
        + ["Unresolved"]
    )

    for row_i, (
        ax,
        threshold,
    ) in enumerate(
        zip(
            axes,
            thresholds,
        )
    ):

        row = pivot.loc[
            threshold
        ]

        total = max(
            float(row.sum()),
            1.0,
        )

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")

        label = (
            row_label_func(
                threshold
            )
            if row_label_func is not None
            else f"{threshold:g}"
        )

        ax.text(
            0.012,
            0.50,
            label,
            ha="left",
            va="center",
            fontsize=19.0 * PLOT_FONT_SCALE,
            fontweight="bold",
            color="#202020",
            zorder=20,
            clip_on=False,
            bbox={
                "boxstyle": "round,pad=0.20",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 1.0,
            },
        )

        if row_i == 0:

            for x, header in zip(
                positions,
                headers,
            ):

                ax.text(
                    x,
                    1.08,
                    header,
                    ha="center",
                    va="bottom",
                    fontsize=17.2 * PLOT_FONT_SCALE,
                    fontweight="bold",
                    color="#252525",
                    linespacing=1.0,
                    clip_on=False,
                )

        _flow_box(
            ax,
            positions[0],
            0.50,
            box_w,
            box_h,
            (
                f"{int(round(total)):,}\n"
                "proteins"
            ),
            facecolor="#F1F1F1",
            edgecolor="#A8A8A8",
            textcolor="#2A2A2A",
            fontsize=16.2 * PLOT_FONT_SCALE,
        )

        remaining = total
        previous_x = positions[0]

        for idx, stage in enumerate(
            model_stages,
            start=1,
        ):

            value = float(
                row.get(
                    stage,
                    0.0,
                )
            )

            input_n = max(
                remaining,
                0.0,
            )

            pct_input = (
                100.0
                * value
                / input_n
                if input_n > 0
                else 0.0
            )

            stage_x = positions[idx]

            _flow_arrow(
                ax,
                previous_x
                + box_w / 2.0,
                0.50,
                stage_x
                - box_w / 2.0,
                0.50,
            )

            _flow_box(
                ax,
                stage_x,
                0.50,
                box_w,
                box_h,
                (
                    f"{int(round(value)):,}\n"
                    f"{_format_flow_percent(pct_input)} "
                    "of input"
                ),
                facecolor=COLOR_MAP.get(
                    stage,
                    "#8C8C8C",
                ),
                fontsize=15.8 * PLOT_FONT_SCALE,
            )

            remaining = max(
                0.0,
                remaining - value,
            )

            next_x = positions[
                idx + 1
            ]

            _flow_arrow(
                ax,
                stage_x
                + box_w / 2.0,
                0.50,
                next_x
                - box_w / 2.0,
                0.50,
                label=(
                    f"{int(round(remaining)):,} "
                    "remaining"
                ),
                label_y_offset=0.26,
            )

            previous_x = stage_x

        unresolved_n = float(
            row.get(
                unresolved_stage,
                remaining,
            )
        )

        _flow_box(
            ax,
            positions[-1],
            0.50,
            box_w,
            box_h,
            (
                f"{int(round(unresolved_n)):,}\n"
                "unresolved"
            ),
            facecolor=COLOR_MAP.get(
                unresolved_stage,
                "#E45756",
            ),
            fontsize=15.8 * PLOT_FONT_SCALE,
        )

    fig.subplots_adjust(
        left=0.045,
        right=0.985,
        top=0.90,
        bottom=0.045,
        hspace=0.40,
    )

    if save_name and SAVE_FIGURES:

        output_path = (
            Path(FIGURE_DIR)
            / Path(str(save_name)).with_suffix(".pdf")
        )

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig.savefig(
            output_path,
            format="pdf",
            bbox_inches="tight",
            pad_inches=0.75,
        )

    plt.show()

    return fig, axes


# =============================================================================
# FULL BASE + scTRANSLATOR FLOW
#
#               /--> 2M -----\
# XGBoost -----+---> 160K ----+----> Unresolved
#               \--> 10K -----/
#
# =============================================================================

def plot_cascade_with_sctranslator_flow_alternative(
    summary,
    *,
    save_name=None,
    row_label_func=None,
    figsize=None,
):
    pivot = _cascade_summary_pivot(
        summary,
        CASCADE_ORDER_WITH_SC,
    )

    thresholds = list(
        pivot.index
    )

    n_rows = len(
        thresholds
    )

    if figsize is None:
        figsize = (
            23.5,
            max(
                9.0,
                3.85 * n_rows,
            ),
        )

    fig, axes = plt.subplots(
        n_rows,
        1,
        figsize=figsize,
        squeeze=False,
    )
    fig.set_size_inches(
        np.asarray(fig.get_size_inches()) * PLOT_CANVAS_SCALE,
        forward=True,
    )

    axes = axes[:, 0]

    base_stages = [
        "Proxy one gene",
        "ENet",
        "XGB",
    ]

    sc_stages = [
        "scTranslator 2M",
        "scTranslator 160K",
        "scTranslator 10K",
    ]

    unresolved_stage = (
        "Still not good after scTranslator"
    )

    # -----------------------------------------------------------------
    # Horizontal placement
    # -----------------------------------------------------------------

    x_start = 0.155
    x_proxy = 0.285
    x_enet = 0.415
    x_xgb = 0.545

    # Three separate scTranslator boxes
    x_sc = 0.735

    # Collector after the three scTranslator branches
    x_merge = 0.845

    # Separate red Unresolved box
    x_unresolved = 0.935

    # -----------------------------------------------------------------
    # Box sizes
    # -----------------------------------------------------------------

    base_w = 0.112
    base_h = 0.43

    # Narrower than previous scTranslator blocks
    sc_w = 0.125
    sc_h = 0.205

    # Narrower red block
    unresolved_w = 0.100
    unresolved_h = 0.235

    # Three scTranslator boxes only
    sc_y = [
        0.74,
        0.50,
        0.26,
    ]

    unresolved_y = 0.50

    for row_i, (
        ax,
        threshold,
    ) in enumerate(
        zip(
            axes,
            thresholds,
        )
    ):

        row = pivot.loc[
            threshold
        ]

        total = max(
            float(row.sum()),
            1.0,
        )

        ax.set_xlim(
            0,
            1,
        )

        ax.set_ylim(
            0,
            1,
        )

        ax.axis(
            "off"
        )

        # -------------------------------------------------------------
        # Threshold label
        # -------------------------------------------------------------

        label = (
            row_label_func(
                threshold
            )
            if row_label_func is not None
            else f"{threshold:g}"
        )

        ax.text(
            0.006,
            0.50,
            label,
            ha="left",
            va="center",
            fontsize=17.5 * PLOT_FONT_SCALE,
            fontweight="bold",
            color="#202020",
            zorder=20,
            clip_on=False,
            bbox={
                "boxstyle": "round,pad=0.16",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 1.0,
            },
        )

        # -------------------------------------------------------------
        # Column headers
        # -------------------------------------------------------------

        if row_i == 0:

            headers = [
                (
                    x_start,
                    "All proteins",
                ),
                (
                    x_proxy,
                    "Same-gene\nproxy",
                ),
                (
                    x_enet,
                    "ElasticNet",
                ),
                (
                    x_xgb,
                    "XGBoost",
                ),
                (
                    x_sc,
                    "scTranslator",
                ),
                (
                    x_unresolved,
                    "Unresolved",
                ),
            ]

            for x, header in headers:

                ax.text(
                    x,
                    1.085,
                    header,
                    ha="center",
                    va="bottom",
                    fontsize=15.8 * PLOT_FONT_SCALE,
                    fontweight="bold",
                    color="#252525",
                    linespacing=1.0,
                    clip_on=False,
                )

        # -------------------------------------------------------------
        # All proteins
        # -------------------------------------------------------------

        _flow_box(
            ax,
            x_start,
            0.50,
            base_w,
            base_h,
            (
                f"{int(round(total)):,}\n"
                "proteins"
            ),
            facecolor="#F1F1F1",
            edgecolor="#A8A8A8",
            textcolor="#2A2A2A",
            fontsize=15.3 * PLOT_FONT_SCALE,
        )

        remaining = total

        base_positions = [
            x_proxy,
            x_enet,
            x_xgb,
        ]

        previous_x = x_start

        # -------------------------------------------------------------
        # Proxy -> ElasticNet -> XGBoost
        # -------------------------------------------------------------

        for stage_index, (
            stage,
            x_stage,
        ) in enumerate(
            zip(
                base_stages,
                base_positions,
            )
        ):

            value = float(
                row.get(
                    stage,
                    0.0,
                )
            )

            input_n = max(
                remaining,
                0.0,
            )

            pct_input = (
                100.0
                * value
                / input_n
                if input_n > 0
                else 0.0
            )

            _flow_arrow(
                ax,
                previous_x
                + base_w / 2.0,
                0.50,
                x_stage
                - base_w / 2.0,
                0.50,
            )

            _flow_box(
                ax,
                x_stage,
                0.50,
                base_w,
                base_h,
                (
                    f"{int(round(value)):,}\n"
                    f"{_format_flow_percent(pct_input)} "
                    "of input"
                ),
                facecolor=COLOR_MAP.get(
                    stage,
                    "#8C8C8C",
                ),
                fontsize=14.8 * PLOT_FONT_SCALE,
            )

            remaining = max(
                0.0,
                remaining - value,
            )

            # Remaining labels only between the base models.
            if stage_index < len(
                base_positions
            ) - 1:

                next_x = base_positions[
                    stage_index + 1
                ]

                _flow_arrow(
                    ax,
                    x_stage
                    + base_w / 2.0,
                    0.50,
                    next_x
                    - base_w / 2.0,
                    0.50,
                    label=(
                        f"{int(round(remaining)):,} "
                        "remaining"
                    ),
                    label_y_offset=0.24,
                )

            previous_x = x_stage

        # =============================================================
        # XGBoost -> THREE scTranslator boxes
        # =============================================================

        xgb_right = (
            x_xgb
            + base_w / 2.0
        )

        sc_left = (
            x_sc
            - sc_w / 2.0
        )

        # Number entering the scTranslator stage.
        ax.text(
            (
                xgb_right
                + sc_left
            )
            / 2.0,
            0.91,
            (
                f"{int(round(remaining)):,} "
                "remaining after XGBoost"
            ),
            ha="center",
            va="center",
            fontsize=13.3 * PLOT_FONT_SCALE,
            fontweight="medium",
            color="#555555",
            zorder=6,
            clip_on=False,
        )

        # -------------------------------------------------------------
        # Draw only the THREE scTranslator model boxes
        # -------------------------------------------------------------

        for stage, y_sc in zip(
            sc_stages,
            sc_y,
        ):

            value = float(
                row.get(
                    stage,
                    0.0,
                )
            )

            # XGBoost fans out to each scTranslator model.
            _flow_arrow(
                ax,
                xgb_right,
                0.50,
                sc_left,
                y_sc,
            )

            rounded_value = int(
                round(value)
            )

            protein_word = (
                "protein"
                if rounded_value == 1
                else "proteins"
            )

            _flow_box(
                ax,
                x_sc,
                y_sc,
                sc_w,
                sc_h,
                (
                    f"{stage}\n"
                    f"{rounded_value:,} "
                    f"{protein_word}"
                ),
                facecolor=COLOR_MAP.get(
                    stage,
                    "#8C8C8C",
                ),
                fontsize=12.7 * PLOT_FONT_SCALE,
            )

        # =============================================================
        # THREE scTranslator outputs -> one common collector
        # =============================================================

        sc_right = (
            x_sc
            + sc_w / 2.0
        )

        # Horizontal lines out of each scTranslator block.
        for y_sc in sc_y:

            _flow_connector(
                ax,
                sc_right,
                y_sc,
                x_merge,
                y_sc,
                linewidth=1.9,
            )

        # Vertical collector joining the three paths.
        _flow_connector(
            ax,
            x_merge,
            min(sc_y),
            x_merge,
            max(sc_y),
            linewidth=1.9,
        )

        # =============================================================
        # Collector -> Unresolved
        # =============================================================

        unresolved_n = float(
            row.get(
                unresolved_stage,
                0.0,
            )
        )

        unresolved_left = (
            x_unresolved
            - unresolved_w / 2.0
        )

        _flow_arrow(
            ax,
            x_merge,
            unresolved_y,
            unresolved_left,
            unresolved_y,
        )

        _flow_box(
            ax,
            x_unresolved,
            unresolved_y,
            unresolved_w,
            unresolved_h,
            (
                f"{int(round(unresolved_n)):,}\n"
                "unresolved"
            ),
            facecolor=COLOR_MAP[
                unresolved_stage
            ],
            fontsize=13.2 * PLOT_FONT_SCALE,
        )

    fig.subplots_adjust(
        left=0.070,
        right=0.992,
        top=0.90,
        bottom=0.030,
        hspace=0.31,
    )

    if save_name and SAVE_FIGURES:

        output_path = (
            Path(FIGURE_DIR)
            / Path(str(save_name)).with_suffix(".pdf")
        )

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig.savefig(
            output_path,
            format="pdf",
            bbox_inches="tight",
            pad_inches=0.44,
        )

    plt.show()

    return fig, axes


# =============================================================================
# scTRANSLATOR-ONLY ALTERNATIVE VIEW
#
# Remaining base-model failures
#         |--> scTranslator 2M   --\
#         |--> scTranslator 160K ---+--> Unresolved
#         \--> scTranslator 10K  --/
# =============================================================================

def plot_sctranslator_branch_alternative(
    rescue_summary,
    *,
    save_name=None,
    row_label_func=None,
    figsize=None,
):
    pivot = _cascade_summary_pivot(
        rescue_summary,
        SC_RESCUE_ORDER,
    )

    thresholds = list(
        pivot.index
    )

    n_rows = len(
        thresholds
    )

    if figsize is None:
        figsize = (
            18.5,
            max(
                8.0,
                3.45 * n_rows,
            ),
        )

    fig, axes = plt.subplots(
        n_rows,
        1,
        figsize=figsize,
        squeeze=False,
    )
    fig.set_size_inches(
        np.asarray(fig.get_size_inches()) * PLOT_CANVAS_SCALE,
        forward=True,
    )

    axes = axes[:, 0]

    sc_stages = [
        "scTranslator 2M",
        "scTranslator 160K",
        "scTranslator 10K",
    ]

    unresolved_stage = (
        "Still not good after scTranslator"
    )

    # -------------------------------------------------------------
    # Layout
    # -------------------------------------------------------------

    x_source = 0.245
    x_sc = 0.595
    x_merge = 0.765
    x_unresolved = 0.905

    source_w = 0.205
    source_h = 0.43

    # Narrower scTranslator boxes
    sc_w = 0.180
    sc_h = 0.205

    # Narrower unresolved block
    unresolved_w = 0.130
    unresolved_h = 0.235

    sc_y = [
        0.74,
        0.50,
        0.26,
    ]

    for row_i, (
        ax,
        threshold,
    ) in enumerate(
        zip(
            axes,
            thresholds,
        )
    ):

        row = pivot.loc[
            threshold
        ]

        total = max(
            float(row.sum()),
            1.0,
        )

        ax.set_xlim(
            0,
            1,
        )

        ax.set_ylim(
            0,
            1,
        )

        ax.axis(
            "off"
        )

        label = (
            row_label_func(
                threshold
            )
            if row_label_func is not None
            else f"{threshold:g}"
        )

        ax.text(
            0.012,
            0.50,
            label,
            ha="left",
            va="center",
            fontsize=17.0 * PLOT_FONT_SCALE,
            fontweight="bold",
            color="#202020",
            clip_on=False,
            bbox={
                "boxstyle": "round,pad=0.18",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 1.0,
            },
        )

        if row_i == 0:

            ax.text(
                x_source,
                1.08,
                "Remaining after base models",
                ha="center",
                va="bottom",
                fontsize=15.2 * PLOT_FONT_SCALE,
                fontweight="bold",
                color="#252525",
                clip_on=False,
            )

            ax.text(
                x_sc,
                1.08,
                "scTranslator",
                ha="center",
                va="bottom",
                fontsize=15.2 * PLOT_FONT_SCALE,
                fontweight="bold",
                color="#252525",
                clip_on=False,
            )

            ax.text(
                x_unresolved,
                1.08,
                "Unresolved",
                ha="center",
                va="bottom",
                fontsize=15.2 * PLOT_FONT_SCALE,
                fontweight="bold",
                color="#252525",
                clip_on=False,
            )

        # -------------------------------------------------------------
        # Source block
        # -------------------------------------------------------------

        _flow_box(
            ax,
            x_source,
            0.50,
            source_w,
            source_h,
            (
                f"{int(round(total)):,}\n"
                "proteins"
            ),
            facecolor="#777777",
            fontsize=15.0 * PLOT_FONT_SCALE,
        )

        source_right = (
            x_source
            + source_w / 2.0
        )

        sc_left = (
            x_sc
            - sc_w / 2.0
        )

        # -------------------------------------------------------------
        # Source -> three scTranslator boxes
        # -------------------------------------------------------------

        for stage, y_sc in zip(
            sc_stages,
            sc_y,
        ):

            value = float(
                row.get(
                    stage,
                    0.0,
                )
            )

            _flow_arrow(
                ax,
                source_right,
                0.50,
                sc_left,
                y_sc,
            )

            rounded_value = int(
                round(value)
            )

            protein_word = (
                "protein"
                if rounded_value == 1
                else "proteins"
            )

            _flow_box(
                ax,
                x_sc,
                y_sc,
                sc_w,
                sc_h,
                (
                    f"{stage}\n"
                    f"{rounded_value:,} "
                    f"{protein_word}"
                ),
                facecolor=COLOR_MAP.get(
                    stage,
                    "#8C8C8C",
                ),
                fontsize=12.9 * PLOT_FONT_SCALE,
            )

        # -------------------------------------------------------------
        # Merge the three scTranslator paths
        # -------------------------------------------------------------

        sc_right = (
            x_sc
            + sc_w / 2.0
        )

        for y_sc in sc_y:

            _flow_connector(
                ax,
                sc_right,
                y_sc,
                x_merge,
                y_sc,
                linewidth=1.9,
            )

        _flow_connector(
            ax,
            x_merge,
            min(sc_y),
            x_merge,
            max(sc_y),
            linewidth=1.9,
        )

        # -------------------------------------------------------------
        # Merge -> Unresolved
        # -------------------------------------------------------------

        unresolved_n = float(
            row.get(
                unresolved_stage,
                0.0,
            )
        )

        unresolved_left = (
            x_unresolved
            - unresolved_w / 2.0
        )

        _flow_arrow(
            ax,
            x_merge,
            0.50,
            unresolved_left,
            0.50,
        )

        _flow_box(
            ax,
            x_unresolved,
            0.50,
            unresolved_w,
            unresolved_h,
            (
                f"{int(round(unresolved_n)):,}\n"
                "unresolved"
            ),
            facecolor=COLOR_MAP[
                unresolved_stage
            ],
            fontsize=13.2 * PLOT_FONT_SCALE,
        )

    fig.subplots_adjust(
        left=0.070,
        right=0.985,
        top=0.90,
        bottom=0.030,
        hspace=0.31,
    )

    if save_name and SAVE_FIGURES:

        output_path = (
            Path(FIGURE_DIR)
            / Path(str(save_name)).with_suffix(".pdf")
        )

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        fig.savefig(
            output_path,
            format="pdf",
            bbox_inches="tight",
            pad_inches=0.44,
        )

    plt.show()

    return fig, axes


# =============================================================================
# ANALYSIS HELPERS REQUIRED BY FOLLOWING CELLS
# =============================================================================

def index_by_protein(df):
    if df is None or len(df) == 0:
        return pd.DataFrame().rename_axis(
            "protein"
        )

    out = df.copy()

    out[
        "protein"
    ] = out[
        "protein"
    ].astype(str)

    return (
        out.drop_duplicates(
            "protein",
            keep="first",
        )
        .set_index(
            "protein"
        )
    )


def metric_is_good(
    value,
    threshold,
    higher,
):
    if not np.isfinite(
        value
    ):
        return False

    return (
        bool(
            value >= threshold
        )
        if higher
        else bool(
            value <= threshold
        )
    )


def best_family_from_results(
    results,
    metric_col,
    family,
    higher,
):
    subset = results.loc[
        results[
            "family_group"
        ]
        == family
    ].copy()

    return best_per_protein(
        subset,
        metric_col,
        higher=higher,
    )


def procedural_metric_cascade(
    threshold,
    *,
    metric_col,
    higher,
    proxy_best,
    enet_best,
    xgb_best,
    universe=None,
):
    universe = pd.Index(
        protein_universe
        if universe is None
        else universe
    ).astype(str)

    remaining = set(
        universe
    )

    rows = []

    stages = [
        (
            "Proxy one gene",
            index_by_protein(
                proxy_best
            ),
        ),
        (
            "ENet",
            index_by_protein(
                enet_best
            ),
        ),
        (
            "XGB",
            index_by_protein(
                xgb_best
            ),
        ),
    ]

    for stage, table in stages:

        for protein in list(
            remaining
        ):

            if protein not in table.index:
                continue

            row = table.loc[
                protein
            ]

            value = row.get(
                metric_col,
                np.nan,
            )

            if metric_is_good(
                value,
                threshold,
                higher,
            ):

                rows.append(
                    {
                        "protein": protein,
                        "threshold": threshold,
                        "stage": stage,
                        "metric": value,
                        "run_label": row.get(
                            "run_label",
                            "",
                        ),
                    }
                )

                remaining.remove(
                    protein
                )

    for protein in sorted(
        remaining
    ):

        rows.append(
            {
                "protein": protein,
                "threshold": threshold,
                "stage": "Still not good",
                "metric": np.nan,
                "run_label": "",
            }
        )

    return pd.DataFrame(
        rows
    )


def set_output_context(mode):
    global FIGURE_DIR
    global EXPORT_TABLE_DIR

    FIGURE_DIR = (
        FIGURE_ROOT
        / mode
    )

    EXPORT_TABLE_DIR = (
        EXPORT_ROOT
        / mode
    )

    FIGURE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    EXPORT_TABLE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    if "uc" in globals():
        uc.set_module_globals(
            globals()
        )


# =============================================================================
# ENET / XGBOOST BEST TABLES
# =============================================================================

model_best = {}

for metric_name, cfg in METRIC_CONFIG.items():

    model_best[
        metric_name
    ] = {
        "enet": best_family_from_results(
            model_results,
            cfg["column"],
            "enet",
            cfg["higher"],
        ),
        "xgb": best_family_from_results(
            model_results,
            cfg["column"],
            "xgb",
            cfg["higher"],
        ),
    }


print(
    "Saved-model best tables"
)

for metric_name, fams in model_best.items():

    print(
        metric_name,
        {
            family: len(df)
            for family, df
            in fams.items()
        },
    )

## 9. Run Pearson, cosine, and MSE cascades for every proxy mode

Each mode uses the exact same saved-model tables and thresholds. Results are stored in `analysis_results[proxy_mode][metric]`, so one mode never overwrites another.

In [ ]:
def run_base_analysis_for_mode(mode):
    set_output_context(mode)
    mode_label = PROXY_MODE_CONFIG[mode]["label"]
    mode_results = all_results_by_mode[mode]
    out = {}

    for metric_name, cfg in METRIC_CONFIG.items():
        proxy_best = best_family_from_results(
            mode_results,
            cfg["column"],
            "proxy",
            cfg["higher"],
        )
        enet_best = model_best[metric_name]["enet"]
        xgb_best = model_best[metric_name]["xgb"]

        def cascade_at(threshold):
            return procedural_metric_cascade(
                threshold,
                metric_col=cfg["column"],
                higher=cfg["higher"],
                proxy_best=proxy_best,
                enet_best=enet_best,
                xgb_best=xgb_best,
            )

        main_assignments = pd.concat(
            [cascade_at(thr) for thr in cfg["main_thresholds"]],
            ignore_index=True,
        )
        main_summary = summarize_assignments(main_assignments, order=CASCADE_ORDER_BASE)

        sweep_assignments = pd.concat(
            [cascade_at(thr) for thr in cfg["sweep_thresholds"]],
            ignore_index=True,
        )
        sweep_summary = summarize_assignments(sweep_assignments, order=CASCADE_ORDER_BASE)

        print(f"\n[{mode}] {cfg['title']} main cascade")
        display(
            main_summary
            .pivot_table(
                index="threshold",
                columns="stage",
                values="proteins",
                fill_value=0,
                aggfunc="sum",
            )
            .reindex(columns=CASCADE_ORDER_BASE)
        )

        # Original stacked-partition artwork (retained, now without bottom notes).
        plot_stacked_partition_rows(
            main_summary,
            order=CASCADE_ORDER_BASE,
            save_name=f"{metric_name}_cascade_030_050_070.pdf",
            row_label_func=cfg["row_label"],
        )

        # Additional, more intuitive stage-by-stage flow artwork.
        plot_cascade_flow_alternative(
            main_summary,
            order=CASCADE_ORDER_BASE,
            save_name=f"{metric_name}_cascade_030_050_070_flow_alternative.pdf",
            row_label_func=cfg["row_label"],
        )

        # Original threshold-sweep artwork (retained, now without bottom notes).
        plot_stacked_partition_rows(
            sweep_summary,
            order=CASCADE_ORDER_BASE,
            save_name=f"{metric_name}_threshold_sweep.pdf",
            figsize=(14, max(6.5, 0.60 * len(cfg["sweep_thresholds"]) + 2.5)),
            row_label_func=cfg["row_label"],
        )

        out[metric_name] = {
            "proxy_best": proxy_best,
            "main_assignments": main_assignments,
            "main_summary": main_summary,
            "sweep_assignments": sweep_assignments,
            "sweep_summary": sweep_summary,
        }

    return out


analysis_results = {
    mode: run_base_analysis_for_mode(mode)
    for mode in PROXY_MODES_TO_RUN
}


## 10. Compare proxy modes directly

The three original notebooks discussed scale mismatch separately. Here the comparison is made in one place using the same proteins and split. Pearson association should be comparatively insensitive to positive affine rescaling, while cosine similarity and MSE can change substantially when predictions are moved onto the protein scale.

In [ ]:
def distribution_diagnostics(name, df):
    arr = np.asarray(df, dtype=float).ravel()
    arr = arr[np.isfinite(arr)]
    return {
        "matrix": name,
        "n_values": int(arr.size),
        "min": float(np.min(arr)),
        "p01": float(np.quantile(arr, 0.01)),
        "median": float(np.median(arr)),
        "mean": float(np.mean(arr)),
        "p99": float(np.quantile(arr, 0.99)),
        "max": float(np.max(arr)),
        "fraction_negative": float(np.mean(arr < 0)),
        "fraction_zero": float(np.mean(arr == 0)),
    }


xy_distribution_diagnostics = pd.DataFrame([
    distribution_diagnostics("Transcriptomics X", X),
    distribution_diagnostics("Proteomics Y", Y),
])

display(xy_distribution_diagnostics)

proxy_metric_diagnostics = (
    proxy_results_all
    .groupby(["proxy_mode", "proxy_mode_label", "normalization"], dropna=False)
    .agg(
        proteins=("protein", "nunique"),
        median_proxy_corr_train=("same_gene_corr_train", "median"),
        median_test_pearson=("test_pearson_corr", "median"),
        median_test_cosine=("test_cosine_similarity", "median"),
        median_test_mse=("test_mse", "median"),
    )
    .reset_index()
)

display(proxy_metric_diagnostics)

comparison_rows = []
for mode in PROXY_MODES_TO_RUN:
    for metric_name in METRIC_CONFIG:
        tmp = analysis_results[mode][metric_name]["main_summary"].copy()
        tmp["proxy_mode"] = mode
        tmp["metric_name"] = metric_name
        comparison_rows.append(tmp)

main_cascade_comparison = pd.concat(comparison_rows, ignore_index=True)
display(main_cascade_comparison)

## 11. Load scTranslator predictions once

scTranslator predictions do not depend on proxy normalization, so they are loaded and evaluated only once on the common H5 test split.

In [ ]:
sctranslator_pred_dfs = load_sctranslator_prediction_tables()
sctranslator_results = evaluate_sctranslator_predictions(
    sctranslator_pred_dfs,
    gt=GT_scaled,
    eval_samples=test_samples,
)

sctranslator_results = add_family_group(sctranslator_results)

print("scTranslator result rows:", len(sctranslator_results))
if len(sctranslator_results):
    display(
        sctranslator_results.groupby("run_label")
        .agg(
            proteins=("protein", "nunique"),
            pearson_median=("test_pearson_corr", "median"),
            pearson_mean=("test_pearson_corr", "mean"),
            cosine_median=("test_cosine_similarity", "median"),
            cosine_mean=("test_cosine_similarity", "mean"),
            mse_median=("test_mse", "median"),
            mse_mean=("test_mse", "mean"),
        )
        .sort_values("pearson_median", ascending=False)
    )
    display(sctranslator_results.head())
else:
    print("No scTranslator predictions were loaded. The final scTranslator plots below will be skipped.")

# Keep moved utility functions synchronized with notebook variables.
try:
    uc.set_module_globals(globals())
except NameError:
    pass

## 12. Generic scTranslator extension analysis

For each proxy mode and each metric, scTranslator is applied only to proteins that remain unresolved after proxy → ENet → XGB. The same generic helper covers Pearson, cosine similarity, and MSE.


In [ ]:
def sctranslator_stage_from_row(row):
    sc_size = row.get("sc_size", np.nan)
    if pd.notna(sc_size) and str(sc_size).strip():
        label = str(sc_size).replace("scTranslator ", "").strip()
    else:
        label = str(row.get("run_label", "")).replace("scTranslator ", "").strip()
    return f"scTranslator {label}" if label else "scTranslator"


def add_sctranslator_rescue(base_assignments, sctranslator_best, *, metric_col, higher):
    sc_idx = index_by_protein(sctranslator_best)
    full_rows = []
    rescue_rows = []

    for _, base_row in base_assignments.iterrows():
        out = base_row.to_dict()
        if str(base_row["stage"]) != "Still not good":
            full_rows.append(out)
            continue

        protein = str(base_row["protein"])
        threshold = float(base_row["threshold"])

        if protein in sc_idx.index:
            sc_row = sc_idx.loc[protein]
            value = sc_row.get(metric_col, np.nan)
            rescued = metric_is_good(value, threshold, higher)
            out.update({
                "stage": sctranslator_stage_from_row(sc_row) if rescued else "Still not good after scTranslator",
                "metric": value,
                metric_col: value,
                "run_label": sc_row.get("run_label", np.nan),
                "sc_size": sc_row.get("sc_size", np.nan),
            })
        else:
            out.update({
                "stage": "Still not good after scTranslator",
                metric_col: np.nan,
                "run_label": np.nan,
                "sc_size": np.nan,
            })

        full_rows.append(out)
        rescue_rows.append(out.copy())

    return pd.DataFrame(full_rows), pd.DataFrame(rescue_rows)


sc_rescue_results = {}
if len(sctranslator_results) > 0:
    sc_best_by_metric = {
        metric_name: best_per_protein(
            sctranslator_results,
            cfg["column"],
            higher=cfg["higher"],
        )
        for metric_name, cfg in METRIC_CONFIG.items()
    }

    for mode in PROXY_MODES_TO_RUN:
        set_output_context(mode)
        mode_label = PROXY_MODE_CONFIG[mode]["label"]
        sc_rescue_results[mode] = {}

        for metric_name, cfg in METRIC_CONFIG.items():
            base_assignments = analysis_results[mode][metric_name]["main_assignments"]
            full, rescue_only = add_sctranslator_rescue(
                base_assignments,
                sc_best_by_metric[metric_name],
                metric_col=cfg["column"],
                higher=cfg["higher"],
            )
            full_summary = summarize_assignments(full, order=CASCADE_ORDER_WITH_SC)
            rescue_summary = summarize_assignments(rescue_only, order=SC_RESCUE_ORDER)

            print(f"\n[{mode}] {cfg['title']} with scTranslator extension")
            display(
                full_summary
                .pivot_table(
                    index="threshold",
                    columns="stage",
                    values="proteins",
                    fill_value=0,
                    aggfunc="sum",
                )
                .reindex(columns=CASCADE_ORDER_WITH_SC)
            )

            # Original full stacked cascade (retained, now without bottom notes).
            plot_stacked_partition_rows(
                full_summary,
                order=CASCADE_ORDER_WITH_SC,
                save_name=f"{metric_name}_cascade_with_sctranslator.pdf",
                row_label_func=cfg["row_label"],
            )

            # Additional full-flow artwork: sequential base cascade, then direct
            # branching from XGBoost into the three scTranslator sizes + unresolved.
            plot_cascade_with_sctranslator_flow_alternative(
                full_summary,
                save_name=f"{metric_name}_cascade_with_sctranslator_flow_alternative.pdf",
                row_label_func=cfg["row_label"],
            )

            # Original scTranslator-only stacked outcome panel (retained).
            plot_stacked_partition_rows(
                rescue_summary,
                order=SC_RESCUE_ORDER,
                save_name=f"sctranslator_{metric_name}_rescue_after_failures.pdf",
                row_label_func=cfg["row_label"],
            )

            # Additional branch-only view of scTranslator outcomes.
            plot_sctranslator_branch_alternative(
                rescue_summary,
                save_name=f"sctranslator_{metric_name}_rescue_branch_alternative.pdf",
                row_label_func=cfg["row_label"],
            )

            sc_rescue_results[mode][metric_name] = {
                "assignments": full,
                "summary": full_summary,
                "rescue_only": rescue_only,
                "rescue_summary": rescue_summary,
            }
else:
    print("Skipped scTranslator extension: no predictions were loaded.")


## 13. Combined publication cascade figure

The three paper bar charts are rendered as one PDF with panel titles, no lettered panel tags, a single color legend, doubled typography, and collision-aware labels for narrow segments. The canonical output is `figures_pdf/cascade_pcc_cos_mse_v1.pdf`; the three previous LaTeX stems are also written as compatibility aliases.

In [ ]:
# =============================================================================
# COMBINED PAPER FIGURE: PEARSON + COSINE + MSE
# =============================================================================
# Final manuscript layout:
# - metric titles reduced by 1/5
# - bars increased in height by 1/3
# - small clean whitespace added before the shared legend
# - in-bar count/percentage labels remain fully contained
# - white in-bar text retains a thin black outline for readability
# - quantitative x-axis ends exactly at the stacked-bar total
# =============================================================================

import matplotlib.patheffects as pe


# -----------------------------------------------------------------------------
# Paper typography
# -----------------------------------------------------------------------------

# Previous title size = 42
# Reduced by ~1/5 -> 34
PAPER_TITLE_FS = 34

PAPER_THRESHOLD_FS = 28
PAPER_INBAR_FS = 22
PAPER_SMALL_LABEL_FS = 18

PAPER_XTICK_FS = 20
PAPER_XLABEL_FS = 26
PAPER_LEGEND_FS = 24


def _paper_panel_pivot(
    summary,
    order,
):
    return _cascade_summary_pivot(
        summary,
        order,
    )


def _draw_paper_cascade_panel(
    ax,
    *,
    summary,
    order,
    title,
    row_label_func,
    show_xlabel=False,
):

    pivot = _paper_panel_pivot(
        summary,
        order,
    )

    thresholds = list(
        pivot.index
    )

    n_rows = len(
        thresholds
    )

    y_positions = (
        np.arange(
            n_rows,
            dtype=float,
        )
        * 2.05
    )

    left = np.zeros(
        n_rows,
        dtype=float,
    )

    totals = (
        pivot
        .sum(axis=1)
        .to_numpy(dtype=float)
    )

    max_total = max(
        float(
            np.nanmax(totals)
        )
        if len(totals)
        else 1.0,
        1.0,
    )

    small_labels = {
        i: []
        for i in range(n_rows)
    }


    # =========================================================================
    # STACKED BARS
    # =========================================================================

    for stage_index, stage in enumerate(
        order
    ):

        values = (
            pivot[stage]
            .to_numpy(dtype=float)
        )

        color = COLOR_MAP.get(
            stage,
            "#8C8C8C",
        )


        bars = ax.barh(
            y_positions,
            values,
            left=left,

            # Previous = 1.02
            # Increased by 1/3:
            height=1.36,

            color=color,
            edgecolor="white",
            linewidth=2.0,
            zorder=3,
        )


        for row_index, (
            bar,
            value,
            total,
        ) in enumerate(
            zip(
                bars,
                values,
                totals,
            )
        ):

            if (
                value <= 0
                or total <= 0
            ):
                continue


            percent = (
                100.0
                * value
                / total
            )


            centre = (
                left[row_index]
                + 0.5
                * value
            )


            # -----------------------------------------------------------------
            # Main segments: count + percentage inside the bar
            # -----------------------------------------------------------------

            if percent >= 8.5:

                text_artist = ax.text(
                    centre,
                    y_positions[row_index],
                    (
                        f"{int(round(value)):,}\n"
                        f"{percent:.1f}%"
                    ),
                    ha="center",
                    va="center",
                    fontsize=PAPER_INBAR_FS,
                    fontweight="bold",
                    color="white",
                    linespacing=0.88,
                    clip_on=False,
                    zorder=6,
                )


                # Thin black contour around white text.
                text_artist.set_path_effects(
                    [
                        pe.withStroke(
                            linewidth=1,
                            foreground="black",
                        )
                    ]
                )


            # -----------------------------------------------------------------
            # Narrow segment: count outside the bar
            # -----------------------------------------------------------------

            else:

                small_labels[
                    row_index
                ].append(
                    {
                        "x": centre,
                        "value": int(
                            round(value)
                        ),
                        "color": color,
                        "stage_index": stage_index,
                    }
                )


        left += values


    # =========================================================================
    # SMALL-SEGMENT ANNOTATIONS
    # =========================================================================

    min_sep = (
        0.040
        * max_total
    )


    for row_index, items in small_labels.items():

        if not items:
            continue


        items = sorted(
            items,
            key=lambda item: item["x"],
        )


        for lane in (
            0,
            1,
        ):

            lane_items = items[
                lane::2
            ]

            if not lane_items:
                continue


            x_text = []


            for item in lane_items:

                candidate = max(
                    item["x"],
                    0.014
                    * max_total,
                )


                if x_text:

                    candidate = max(
                        candidate,
                        x_text[-1]
                        + min_sep,
                    )


                x_text.append(
                    candidate
                )


            right_limit = (
                0.985
                * max_total
            )


            overflow = max(
                0.0,
                x_text[-1]
                - right_limit,
            )


            x_text = [
                max(
                    0.010
                    * max_total,
                    x
                    - overflow,
                )
                for x in x_text
            ]


            # Slightly larger vertical displacement because bars are taller.
            y_offset = (
                -0.88
                if lane == 0
                else 0.88
            )


            for item, label_x in zip(
                lane_items,
                x_text,
            ):

                ax.annotate(
                    f"{item['value']:,}",

                    xy=(
                        item["x"],
                        y_positions[
                            row_index
                        ],
                    ),

                    xytext=(
                        label_x,
                        y_positions[
                            row_index
                        ]
                        + y_offset,
                    ),

                    ha="center",
                    va="center",

                    fontsize=PAPER_SMALL_LABEL_FS,
                    fontweight="bold",
                    color="#222222",

                    arrowprops={
                        "arrowstyle": "-",
                        "color": item[
                            "color"
                        ],
                        "linewidth": 1.8,
                        "shrinkA": 2,
                        "shrinkB": 2,
                    },

                    clip_on=False,
                    zorder=8,
                )


    # =========================================================================
    # AXES
    # =========================================================================

    ax.set_yticks(
        y_positions
    )


    ax.set_yticklabels(
        [
            row_label_func(t)
            for t in thresholds
        ],
        fontsize=PAPER_THRESHOLD_FS,
    )


    ax.set_title(
        title,
        fontsize=PAPER_TITLE_FS,
        fontweight="bold",
        pad=24,
    )


    # Exact bar extent.
    ax.set_xlim(
        0,
        max_total,
    )


    # Slightly more room for labels outside taller bars.
    ax.set_ylim(
        y_positions[-1]
        + 1.45,
        -1.45,
    )


    ax.xaxis.set_major_locator(
        MaxNLocator(
            integer=True,
            nbins=7,
        )
    )


    ax.tick_params(
        axis="x",
        labelsize=PAPER_XTICK_FS,
        width=1.6,
        length=7,
        pad=6,
    )


    ax.tick_params(
        axis="y",
        length=0,
        pad=12,
    )


    ax.grid(
        axis="x",
        alpha=0.20,
        linewidth=1.2,
        zorder=0,
    )


    ax.grid(
        axis="y",
        visible=False,
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )


    ax.spines[
        "right"
    ].set_visible(
        False
    )


    ax.spines[
        "left"
    ].set_visible(
        False
    )


    ax.set_xlabel(
        (
            "Number of proteins"
            if show_xlabel
            else ""
        ),
        fontsize=PAPER_XLABEL_FS,
        labelpad=14,
    )


    return pivot


# =============================================================================
# BUILD PAPER PANELS
# =============================================================================

PAPER_PROXY_MODE = (
    "train_zscore"
)


if PAPER_PROXY_MODE not in analysis_results:

    raise RuntimeError(
        f"Missing paper proxy mode: {PAPER_PROXY_MODE}"
    )


paper_specs = []


for metric_name, title in [
    (
        "pearson",
        "Pearson correlation",
    ),
    (
        "cosine",
        "Cosine similarity",
    ),
]:

    if (
        PAPER_PROXY_MODE
        in sc_rescue_results

        and metric_name
        in sc_rescue_results[
            PAPER_PROXY_MODE
        ]
    ):

        summary = (
            sc_rescue_results[
                PAPER_PROXY_MODE
            ][
                metric_name
            ][
                "summary"
            ]
        )

        order = (
            CASCADE_ORDER_WITH_SC
        )

    else:

        summary = (
            analysis_results[
                PAPER_PROXY_MODE
            ][
                metric_name
            ][
                "main_summary"
            ]
        )

        order = (
            CASCADE_ORDER_BASE
        )


    paper_specs.append(
        (
            title,
            summary,
            order,
            METRIC_CONFIG[
                metric_name
            ][
                "row_label"
            ],
        )
    )


paper_specs.append(
    (
        "Mean squared error",

        analysis_results[
            PAPER_PROXY_MODE
        ][
            "mse"
        ][
            "main_summary"
        ],

        CASCADE_ORDER_BASE,

        METRIC_CONFIG[
            "mse"
        ][
            "row_label"
        ],
    )
)


# =============================================================================
# FIGURE
# =============================================================================

fig, axes = plt.subplots(
    3,
    1,
    figsize=(
        26,
        32,
    ),
    constrained_layout=False,
)


pivots = []


for panel_index, (
    ax,
    spec,
) in enumerate(
    zip(
        axes,
        paper_specs,
    )
):

    (
        title,
        summary,
        order,
        row_label_func,
    ) = spec


    pivots.append(
        _draw_paper_cascade_panel(
            ax,
            summary=summary,
            order=order,
            title=title,
            row_label_func=row_label_func,
            show_xlabel=(
                panel_index
                == len(
                    paper_specs
                )
                - 1
            ),
        )
    )


# =============================================================================
# ONE SHARED LEGEND
# =============================================================================

legend_stages = []


for stage in CASCADE_ORDER_WITH_SC:

    if stage.startswith(
        "Still not good"
    ):
        continue


    if any(
        stage in pivot.columns
        and float(
            pivot[
                stage
            ].sum()
        ) > 0
        for pivot in pivots
    ):

        legend_stages.append(
            stage
        )


legend_handles = [
    Patch(
        facecolor=COLOR_MAP[
            stage
        ],
        edgecolor="white",
        label=CASCADE_LEGEND_LABELS[
            stage
        ],
    )
    for stage in legend_stages
]


legend_handles.append(
    Patch(
        facecolor=COLOR_MAP[
            "Still not good"
        ],
        edgecolor="white",
        label=(
            "Unresolved - no tested model "
            "reaches the threshold"
        ),
    )
)


fig.legend(
    handles=legend_handles,
    loc="lower center",

    # Slightly lower than before, creating a clean white gap.
    bbox_to_anchor=(
        0.5,
        0.018,
    ),

    ncol=2,
    frameon=False,

    fontsize=PAPER_LEGEND_FS,

    handlelength=1.8,
    handleheight=1.05,
    handletextpad=0.75,

    columnspacing=1.5,
    labelspacing=0.80,
)


# Slightly larger bottom reservation:
# previous = 0.17
# new = 0.205
# This creates a little white space between the figure and legend.
fig.subplots_adjust(
    left=0.19,
    right=0.985,
    top=0.95,
    bottom=0.205,
    hspace=0.42,
)


# =============================================================================
# EXPORT
# =============================================================================

combined_names = [
    "cascade_pcc_cos_mse_v1.pdf",

    "cascade_pcc_v1.pdf",
    "cascade_cos_v1.pdf",
    "cascade_mse_v1.pdf",
]


for combined_name in combined_names:

    fig.savefig(
        FIGURE_ROOT
        / combined_name,

        format="pdf",

        bbox_inches="tight",

        pad_inches=0.35,

        metadata={
            "Title": (
                "CCLE PCC, cosine and MSE cascades"
            )
        },
    )


COMBINED_CASCADE_PDF = (
    FIGURE_ROOT
    / combined_names[
        0
    ]
)


print(
    "Saved combined paper cascade:",
    COMBINED_CASCADE_PDF,
)


plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# COMBINED PAPER FIGURE: PEARSON + COSINE + MSE
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator


# =============================================================================
# PAPER TYPOGRAPHY
# =============================================================================

PAPER_TITLE_FS = 34

PAPER_THRESHOLD_FS = 28
PAPER_INBAR_FS = 22
PAPER_SMALL_LABEL_FS = 18

PAPER_XTICK_FS = 20
PAPER_XLABEL_FS = 26
PAPER_LEGEND_FS = 27


# =============================================================================
# Y-AXIS LABEL ALIGNMENT
# =============================================================================
#
# All Pearson / Cosine / MSE row labels have the same LEFT edge.
#
# More negative -> labels further left
# Less negative -> labels further right
#
# =============================================================================

PAPER_YLABEL_X = -0.19


# =============================================================================
# HELPERS
# =============================================================================

def _paper_panel_pivot(
    summary,
    order,
):
    return _cascade_summary_pivot(
        summary,
        order,
    )


def _draw_paper_cascade_panel(
    ax,
    *,
    summary,
    order,
    title,
    row_label_func,
    show_xlabel=False,
):

    pivot = _paper_panel_pivot(
        summary,
        order,
    )

    thresholds = list(
        pivot.index
    )

    n_rows = len(
        thresholds
    )

    y_positions = (
        np.arange(
            n_rows,
            dtype=float,
        )
        * 2.05
    )

    left = np.zeros(
        n_rows,
        dtype=float,
    )

    totals = (
        pivot
        .sum(axis=1)
        .to_numpy(dtype=float)
    )

    max_total = max(
        float(
            np.nanmax(totals)
        )
        if len(totals)
        else 1.0,
        1.0,
    )

    small_labels = {
        i: []
        for i in range(n_rows)
    }


    # =========================================================================
    # STACKED BARS
    # =========================================================================

    for stage_index, stage in enumerate(
        order
    ):

        values = (
            pivot[stage]
            .to_numpy(dtype=float)
        )

        color = COLOR_MAP.get(
            stage,
            "#8C8C8C",
        )

        bars = ax.barh(
            y_positions,
            values,
            left=left,

            height=1.36,

            color=color,
            edgecolor="white",
            linewidth=2.0,
            zorder=3,
        )


        for row_index, (
            bar,
            value,
            total,
        ) in enumerate(
            zip(
                bars,
                values,
                totals,
            )
        ):

            if (
                value <= 0
                or total <= 0
            ):
                continue

            percent = (
                100.0
                * value
                / total
            )

            centre = (
                left[row_index]
                + 0.5
                * value
            )


            # -----------------------------------------------------------------
            # Main segments:
            # count + percentage inside bar
            # -----------------------------------------------------------------

            if percent >= 8.5:

                text_artist = ax.text(
                    centre,
                    y_positions[row_index],
                    (
                        f"{int(round(value)):,}\n"
                        f"{percent:.1f}%"
                    ),
                    ha="center",
                    va="center",
                    fontsize=PAPER_INBAR_FS,
                    fontweight="bold",
                    color="white",
                    linespacing=0.88,
                    clip_on=False,
                    zorder=6,
                )

                text_artist.set_path_effects(
                    [
                        pe.withStroke(
                            linewidth=1,
                            foreground="black",
                        )
                    ]
                )


            # -----------------------------------------------------------------
            # Narrow segments:
            # count outside bar
            # -----------------------------------------------------------------

            else:

                small_labels[
                    row_index
                ].append(
                    {
                        "x": centre,
                        "value": int(
                            round(value)
                        ),
                        "color": color,
                        "stage_index": stage_index,
                    }
                )

        left += values


    # =========================================================================
    # SMALL-SEGMENT ANNOTATIONS
    # =========================================================================

    min_sep = (
        0.040
        * max_total
    )

    for row_index, items in small_labels.items():

        if not items:
            continue

        items = sorted(
            items,
            key=lambda item: item["x"],
        )

        for lane in (
            0,
            1,
        ):

            lane_items = items[
                lane::2
            ]

            if not lane_items:
                continue

            x_text = []

            for item in lane_items:

                candidate = max(
                    item["x"],
                    0.014
                    * max_total,
                )

                if x_text:

                    candidate = max(
                        candidate,
                        x_text[-1]
                        + min_sep,
                    )

                x_text.append(
                    candidate
                )

            right_limit = (
                0.985
                * max_total
            )

            overflow = max(
                0.0,
                x_text[-1]
                - right_limit,
            )

            x_text = [
                max(
                    0.010
                    * max_total,
                    x
                    - overflow,
                )
                for x in x_text
            ]

            y_offset = (
                -0.88
                if lane == 0
                else 0.88
            )

            for item, label_x in zip(
                lane_items,
                x_text,
            ):

                ax.annotate(
                    f"{item['value']:,}",

                    xy=(
                        item["x"],
                        y_positions[
                            row_index
                        ],
                    ),

                    xytext=(
                        label_x,
                        y_positions[
                            row_index
                        ]
                        + y_offset,
                    ),

                    ha="center",
                    va="center",

                    fontsize=PAPER_SMALL_LABEL_FS,
                    fontweight="bold",
                    color="#222222",

                    arrowprops={
                        "arrowstyle": "-",
                        "color": item[
                            "color"
                        ],
                        "linewidth": 1.8,
                        "shrinkA": 2,
                        "shrinkB": 2,
                    },

                    clip_on=False,
                    zorder=8,
                )


    # =========================================================================
    # AXES
    # =========================================================================

    ax.set_yticks(
        y_positions
    )

    ax.set_yticklabels(
        [
            row_label_func(t)
            for t in thresholds
        ],
        fontsize=PAPER_THRESHOLD_FS,
    )


    # -------------------------------------------------------------------------
    # Left-align all metric labels at exactly the same x position
    # -------------------------------------------------------------------------

    for label in ax.get_yticklabels():

        label.set_horizontalalignment(
            "left"
        )

        label.set_x(
            PAPER_YLABEL_X
        )


    ax.set_title(
        title,
        fontsize=PAPER_TITLE_FS,
        fontweight="bold",
        pad=24,
    )


    # Exact stacked-bar extent

    ax.set_xlim(
        0,
        max_total,
    )


    ax.set_ylim(
        y_positions[-1]
        + 1.45,
        -1.45,
    )


    ax.xaxis.set_major_locator(
        MaxNLocator(
            integer=True,
            nbins=7,
        )
    )


    ax.tick_params(
        axis="x",
        labelsize=PAPER_XTICK_FS,
        width=1.6,
        length=7,
        pad=6,
    )


    ax.tick_params(
        axis="y",
        length=0,
        pad=12,
    )


    ax.grid(
        axis="x",
        alpha=0.20,
        linewidth=1.2,
        zorder=0,
    )

    ax.grid(
        axis="y",
        visible=False,
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )

    ax.spines[
        "left"
    ].set_visible(
        False
    )


    ax.set_xlabel(
        (
            "Number of proteins"
            if show_xlabel
            else ""
        ),
        fontsize=PAPER_XLABEL_FS,
        labelpad=14,
    )


    return pivot


# =============================================================================
# BUILD PAPER PANELS
# =============================================================================

PAPER_PROXY_MODE = (
    "train_zscore"
)


if PAPER_PROXY_MODE not in analysis_results:

    raise RuntimeError(
        f"Missing paper proxy mode: {PAPER_PROXY_MODE}"
    )


paper_specs = []


for metric_name, title in [
    (
        "pearson",
        "Pearson correlation",
    ),
    (
        "cosine",
        "Cosine similarity",
    ),
]:

    if (
        PAPER_PROXY_MODE
        in sc_rescue_results

        and metric_name
        in sc_rescue_results[
            PAPER_PROXY_MODE
        ]
    ):

        summary = (
            sc_rescue_results[
                PAPER_PROXY_MODE
            ][
                metric_name
            ][
                "summary"
            ]
        )

        order = (
            CASCADE_ORDER_WITH_SC
        )

    else:

        summary = (
            analysis_results[
                PAPER_PROXY_MODE
            ][
                metric_name
            ][
                "main_summary"
            ]
        )

        order = (
            CASCADE_ORDER_BASE
        )


    paper_specs.append(
        (
            title,
            summary,
            order,
            METRIC_CONFIG[
                metric_name
            ][
                "row_label"
            ],
        )
    )


paper_specs.append(
    (
        "Mean squared error",

        analysis_results[
            PAPER_PROXY_MODE
        ][
            "mse"
        ][
            "main_summary"
        ],

        CASCADE_ORDER_BASE,

        METRIC_CONFIG[
            "mse"
        ][
            "row_label"
        ],
    )
)


# =============================================================================
# FIGURE
# =============================================================================

fig, axes = plt.subplots(
    3,
    1,
    figsize=(
        26,
        32,
    ),
    constrained_layout=False,
)


pivots = []


for panel_index, (
    ax,
    spec,
) in enumerate(
    zip(
        axes,
        paper_specs,
    )
):

    (
        title,
        summary,
        order,
        row_label_func,
    ) = spec


    pivots.append(
        _draw_paper_cascade_panel(
            ax,
            summary=summary,
            order=order,
            title=title,
            row_label_func=row_label_func,
            show_xlabel=(
                panel_index
                == len(
                    paper_specs
                )
                - 1
            ),
        )
    )


# =============================================================================
# FINAL FIGURE GEOMETRY
# =============================================================================

fig.subplots_adjust(
    left=0.19,
    right=0.985,
    top=0.965,
    bottom=0.155,
    hspace=0.30,
)


# =============================================================================
# ONE SHARED LEGEND
# =============================================================================
#
# EXACT STRUCTURE:
#
# COLUMN 1               COLUMN 2                 COLUMN 3
#
# Same-gene proxy        scTranslator 10K         Unresolved proteins
# ElasticNet             scTranslator 160K
# XGBoost                scTranslator 2M
#
# =============================================================================


def _normalise_legend_name(text):

    return "".join(
        ch.lower()
        for ch in str(text)
        if ch.isalnum()
    )


def _short_legend_label(label):

    label = str(
        label
    ).strip()

    if "—" in label:

        label = label.split(
            "—",
            1,
        )[0].strip()

    elif "--" in label:

        label = label.split(
            "--",
            1,
        )[0].strip()

    return label


def _find_legend_stage(
    model_name
):

    target = (
        _normalise_legend_name(
            model_name
        )
    )


    # Explicit legend-label mapping

    for stage, raw_label in (
        CASCADE_LEGEND_LABELS.items()
    ):

        short_label = (
            _short_legend_label(
                raw_label
            )
        )

        if (
            _normalise_legend_name(
                short_label
            )
            == target
        ):

            return stage


    # Fallback to cascade stage name

    for stage in (
        CASCADE_ORDER_WITH_SC
    ):

        if (
            _normalise_legend_name(
                stage
            )
            == target
        ):

            return stage


    available = [
        _short_legend_label(
            label
        )
        for label in
        CASCADE_LEGEND_LABELS.values()
    ]


    raise RuntimeError(
        f"Could not find legend stage "
        f"for {model_name!r}. "
        f"Available labels: {available}"
    )


def _model_legend_patch(
    model_name
):

    stage = (
        _find_legend_stage(
            model_name
        )
    )

    return Patch(
        facecolor=COLOR_MAP[
            stage
        ],
        edgecolor="white",
        label=model_name,
    )


# =============================================================================
# LEGEND HANDLES
# =============================================================================
#
# Matplotlib fills each COLUMN vertically:
#
# 1-3 -> column 1
# 4-6 -> column 2
# 7-9 -> column 3
#
# Therefore Unresolved proteins is handle 7 and appears
# at the TOP of column 3.
# =============================================================================

legend_handles = [

    # -------------------------------------------------------------------------
    # COLUMN 1
    # -------------------------------------------------------------------------

    _model_legend_patch(
        "Same-gene proxy"
    ),

    _model_legend_patch(
        "ElasticNet"
    ),

    _model_legend_patch(
        "XGBoost"
    ),


    # -------------------------------------------------------------------------
    # COLUMN 2
    # -------------------------------------------------------------------------

    _model_legend_patch(
        "scTranslator 10K"
    ),

    _model_legend_patch(
        "scTranslator 160K"
    ),

    _model_legend_patch(
        "scTranslator 2M"
    ),


    # -------------------------------------------------------------------------
    # COLUMN 3
    # -------------------------------------------------------------------------

    Patch(
        facecolor=COLOR_MAP[
            "Still not good"
        ],
        edgecolor="white",
        label="Unresolved proteins",
    ),

    Patch(
        facecolor="none",
        edgecolor="none",
        alpha=0.0,
        label="",
    ),

    Patch(
        facecolor="none",
        edgecolor="none",
        alpha=0.0,
        label="",
    ),
]


# =============================================================================
# LEGEND HORIZONTAL ALIGNMENT
# =============================================================================
#
# IMPORTANT:
#
# The bars begin exactly at the left edge of the plotting axes.
#
# Therefore the legend must use the SAME figure-space x coordinate:
#
#     axes[0].get_position().x0
#
# The metric-label position is NOT used here.
# This means metric labels can move left/right without moving the legend.
#
# borderpad=0 removes the internal left padding so that the first coloured
# legend patch begins at the same x coordinate as the bars.
# =============================================================================

axes_bbox = axes[0].get_position()

legend_left = axes_bbox.x0


fig.legend(
    handles=legend_handles,

    loc="lower left",

    bbox_to_anchor=(
        legend_left,
        0.018,
    ),

    bbox_transform=fig.transFigure,

    ncol=3,

    frameon=False,

    fontsize=PAPER_LEGEND_FS,

    handlelength=1.8,
    handleheight=1.05,
    handletextpad=0.75,

    columnspacing=2.0,
    labelspacing=0.80,

    # Critical for exact alignment with bar start
    borderaxespad=0.0,
    borderpad=0.0,
)


# =============================================================================
# EXPORT
# =============================================================================

combined_names = [

    "cascade_pcc_cos_mse_v1.pdf",

    "cascade_pcc_v1.pdf",
    "cascade_cos_v1.pdf",
    "cascade_mse_v1.pdf",
]


for combined_name in combined_names:

    fig.savefig(
        FIGURE_ROOT
        / combined_name,

        format="pdf",

        bbox_inches="tight",

        pad_inches=0.35,

        metadata={
            "Title": (
                "CCLE PCC, cosine and MSE cascades"
            )
        },
    )


COMBINED_CASCADE_PDF = (
    FIGURE_ROOT
    / combined_names[
        0
    ]
)


print(
    "Saved combined paper cascade:",
    COMBINED_CASCADE_PDF,
)


plt.show()
plt.close(fig)

In [ ]:
# =============================================================================
# COMBINED PAPER FIGURE V2: PEARSON + COSINE + MSE FLOW VIEW
# =============================================================================
# This is the combined version of the alternative stage-by-stage cascade artwork.
# It is drawn directly from the underlying cascade summaries (no PNG dependency),
# uses one shared legend, and is exported only as vector PDF.


def _draw_paper_flow_row(
    ax,
    *,
    row,
    threshold,
    row_label_func,
    show_headers=False,
    section_title=None,
):
    model_stages = ["Proxy one gene", "ENet", "XGB"]
    positions = np.linspace(0.245, 0.935, 5)
    box_w = 0.132
    box_h = 0.44
    headers = ["All proteins", "Same-gene\nproxy", "ElasticNet", "XGBoost", "Unresolved"]

    total = max(float(row.sum()), 1.0)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    if section_title:
        ax.text(
            0.0, 1.42, section_title,
            transform=ax.transAxes,
            ha="left", va="bottom",
            fontsize=28, fontweight="bold",
            color="#202020", clip_on=False,
        )

    ax.text(
        0.005, 0.50, row_label_func(threshold),
        ha="left", va="center",
        fontsize=21, fontweight="bold",
        color="#202020", clip_on=False,
        bbox={"boxstyle": "round,pad=0.18", "facecolor": "white", "edgecolor": "none"},
    )

    if show_headers:
        for x, header in zip(positions, headers):
            ax.text(
                x, 1.10, header,
                ha="center", va="bottom",
                fontsize=20, fontweight="bold",
                color="#252525", linespacing=1.0,
                clip_on=False,
            )

    _flow_box(
        ax, positions[0], 0.50, box_w, box_h,
        f"{int(round(total)):,}\nproteins",
        facecolor="#F1F1F1", edgecolor="#A8A8A8",
        textcolor="#2A2A2A", fontsize=19,
    )

    remaining = total
    previous_x = positions[0]
    for idx, stage in enumerate(model_stages, start=1):
        value = float(row.get(stage, 0.0))
        input_n = max(remaining, 0.0)
        pct_input = 100.0 * value / input_n if input_n > 0 else 0.0
        stage_x = positions[idx]

        _flow_arrow(
            ax,
            previous_x + box_w / 2.0, 0.50,
            stage_x - box_w / 2.0, 0.50,
        )
        _flow_box(
            ax, stage_x, 0.50, box_w, box_h,
            f"{int(round(value)):,}\n{_format_flow_percent(pct_input)} of input",
            facecolor=COLOR_MAP.get(stage, "#8C8C8C"),
            fontsize=18,
        )

        remaining = max(0.0, remaining - value)
        next_x = positions[idx + 1]
        _flow_arrow(
            ax,
            stage_x + box_w / 2.0, 0.50,
            next_x - box_w / 2.0, 0.50,
            label=f"{int(round(remaining)):,} remaining",
            label_y_offset=0.28,
        )
        previous_x = stage_x

    unresolved_n = float(row.get("Still not good", remaining))
    _flow_box(
        ax, positions[-1], 0.50, box_w, box_h,
        f"{int(round(unresolved_n)):,}\nunresolved",
        facecolor=COLOR_MAP.get("Still not good", "#E45756"),
        fontsize=18,
    )


_v2_specs = [
    ("pearson", "Pearson correlation"),
    ("cosine", "Cosine similarity"),
    ("mse", "Mean squared error"),
]

_v2_rows = []
for metric_name, section_title in _v2_specs:
    summary = analysis_results[PAPER_PROXY_MODE][metric_name]["main_summary"]
    pivot = _cascade_summary_pivot(summary, CASCADE_ORDER_BASE)
    _v2_rows.append((metric_name, section_title, pivot))

n_total_rows = sum(len(pivot.index) for _, _, pivot in _v2_rows)
fig_v2, axes_v2 = plt.subplots(
    n_total_rows, 1,
    figsize=(30, max(30, 3.8 * n_total_rows)),
    squeeze=False,
)
axes_v2 = axes_v2[:, 0]

axis_i = 0
for metric_name, section_title, pivot in _v2_rows:
    thresholds = list(pivot.index)
    for row_i, threshold in enumerate(thresholds):
        _draw_paper_flow_row(
            axes_v2[axis_i],
            row=pivot.loc[threshold],
            threshold=threshold,
            row_label_func=METRIC_CONFIG[metric_name]["row_label"],
            show_headers=(row_i == 0),
            section_title=(section_title if row_i == 0 else None),
        )
        axis_i += 1

v2_legend_handles = [
    Patch(facecolor=COLOR_MAP["Proxy one gene"], edgecolor="white", label="Same-gene proxy"),
    Patch(facecolor=COLOR_MAP["ENet"], edgecolor="white", label="ElasticNet"),
    Patch(facecolor=COLOR_MAP["XGB"], edgecolor="white", label="XGBoost"),
    Patch(facecolor=COLOR_MAP["Still not good"], edgecolor="white", label="Unresolved"),
]
fig_v2.legend(
    handles=v2_legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.012),
    ncol=4,
    frameon=False,
    fontsize=20,
    handlelength=1.8,
    columnspacing=2.2,
)
fig_v2.subplots_adjust(left=0.055, right=0.99, top=0.965, bottom=0.085, hspace=0.58)

combined_v2_names = [
    "cascade_pcc_cos_mse_v2.pdf",
    "cascade_pcc_v2.pdf",
    "cascade_cos_v2.pdf",
    "cascade_mse_v2.pdf",
]
for combined_name in combined_v2_names:
    fig_v2.savefig(
        FIGURE_ROOT / combined_name,
        format="pdf",
        bbox_inches="tight",
        pad_inches=0.55,
        metadata={"Title": "CCLE PCC, cosine and MSE cascade flow view"},
    )

COMBINED_CASCADE_V2_PDF = FIGURE_ROOT / combined_v2_names[0]
print("Saved combined V2 paper cascade:", COMBINED_CASCADE_V2_PDF)
plt.show()
plt.close(fig_v2)


## 14. Optional exports

Exports are now mode-aware, so the three proxy variants cannot overwrite one another. `SAVE_OUTPUTS` controls the full result/summary CSVs. `SAVE_CASCADE_PROTEIN_EXPORTS` retains the original one-file-per-partition behavior, but writes a separate folder for each proxy mode.

In [ ]:
def safe_name(value):
    text = str(value).strip()
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text.strip("_") or "value"


def assignment_tables_for_mode(mode):
    tables = []
    for metric_name in METRIC_CONFIG:
        base = analysis_results[mode][metric_name]
        tables.extend([
            (f"main_{metric_name}_cascade", base["main_assignments"]),
            (f"{metric_name}_threshold_sweep", base["sweep_assignments"]),
        ])
        if mode in sc_rescue_results and metric_name in sc_rescue_results[mode]:
            sc = sc_rescue_results[mode][metric_name]
            tables.extend([
                (f"{metric_name}_cascade_with_sctranslator", sc["assignments"]),
                (f"sctranslator_{metric_name}_rescue_only", sc["rescue_only"]),
            ])
    return tables


if SAVE_OUTPUTS:
    for mode in PROXY_MODES_TO_RUN:
        set_output_context(mode)
        proxy_results_by_mode[mode].to_csv(EXPORT_TABLE_DIR / "proxy_results_all_same_gene.csv", index=False)
        all_results_by_mode[mode].to_csv(EXPORT_TABLE_DIR / "all_results_proxy_enet_xgb.csv", index=False)

        for metric_name in METRIC_CONFIG:
            base = analysis_results[mode][metric_name]
            base["main_assignments"].to_csv(
                EXPORT_TABLE_DIR / f"{metric_name}_cascade_assignments_030_050_070.csv", index=False
            )
            base["main_summary"].to_csv(
                EXPORT_TABLE_DIR / f"{metric_name}_cascade_summary_030_050_070.csv", index=False
            )
            base["sweep_assignments"].to_csv(
                EXPORT_TABLE_DIR / f"{metric_name}_threshold_sweep_assignments.csv", index=False
            )
            base["sweep_summary"].to_csv(
                EXPORT_TABLE_DIR / f"{metric_name}_threshold_sweep_summary.csv", index=False
            )

            if mode in sc_rescue_results and metric_name in sc_rescue_results[mode]:
                sc = sc_rescue_results[mode][metric_name]
                sc["assignments"].to_csv(
                    EXPORT_TABLE_DIR / f"{metric_name}_cascade_with_sctranslator_assignments.csv", index=False
                )
                sc["summary"].to_csv(
                    EXPORT_TABLE_DIR / f"{metric_name}_cascade_with_sctranslator_summary.csv", index=False
                )
                sc["rescue_only"].to_csv(
                    EXPORT_TABLE_DIR / f"sctranslator_{metric_name}_rescue_only_assignments.csv", index=False
                )
                sc["rescue_summary"].to_csv(
                    EXPORT_TABLE_DIR / f"sctranslator_{metric_name}_rescue_only_summary.csv", index=False
                )

    if len(sctranslator_results) > 0:
        sctranslator_results.to_csv(EXPORT_ROOT / "sctranslator_metrics.csv", index=False)
    print("Saved full outputs under:", EXPORT_ROOT)
else:
    print("SAVE_OUTPUTS is False; full result/summary CSV export skipped.")


if SAVE_CASCADE_PROTEIN_EXPORTS:
    for mode in PROXY_MODES_TO_RUN:
        mode_export_dir = EXPORT_ROOT / mode / "protein_names_by_cascade_partition"
        split_export_dir = mode_export_dir / "protein_lists_one_file_per_partition"
        split_export_dir.mkdir(parents=True, exist_ok=True)

        long_parts = []
        for analysis_name, df in assignment_tables_for_mode(mode):
            if df is None or len(df) == 0:
                continue
            tmp = df.copy()
            tmp["dataset"] = "CCLE"
            tmp["proxy_mode"] = mode
            tmp["analysis"] = analysis_name
            tmp["protein"] = tmp["protein"].astype(str)
            long_parts.append(tmp)

        if not long_parts:
            print(f"[{mode}] No assignment tables available for protein-list export.")
            continue

        long_df = pd.concat(long_parts, ignore_index=True, sort=False)
        group_cols = ["dataset", "proxy_mode", "analysis", "threshold", "stage"]

        grouped_df = (
            long_df
            .groupby(group_cols, dropna=False)
            .agg(
                n_proteins=("protein", "nunique"),
                protein_names=("protein", lambda x: "; ".join(sorted(set(map(str, x))))),
            )
            .reset_index()
            .sort_values(group_cols)
            .reset_index(drop=True)
        )

        manifest_rows = []
        for keys, sub in long_df.groupby(group_cols, dropna=False):
            info = dict(zip(group_cols, keys))
            file_name = "__".join([
                safe_name(info["dataset"]),
                safe_name(info["proxy_mode"]),
                safe_name(info["analysis"]),
                f"threshold_{safe_name(info['threshold'])}",
                safe_name(info["stage"]),
            ]) + ".csv"
            out_path = split_export_dir / file_name

            keep_cols = [
                c for c in [
                    "dataset", "proxy_mode", "analysis", "threshold", "stage", "protein",
                    "metric", "test_pearson_corr", "test_mse", "test_cosine_similarity",
                    "same_gene_corr_train", "same_gene_corr_full_reference_only",
                    "run_label", "sc_size",
                ]
                if c in sub.columns
            ]
            split_df = sub[keep_cols].drop_duplicates().sort_values("protein").reset_index(drop=True)
            split_df.to_csv(out_path, index=False)

            manifest_rows.append({
                **info,
                "n_proteins": split_df["protein"].nunique(),
                "file_name": file_name,
                "relative_path": str(out_path.relative_to(mode_export_dir)),
            })

        manifest_df = pd.DataFrame(manifest_rows)
        long_df.to_csv(mode_export_dir / "cascade_partition_proteins_long.csv", index=False)
        grouped_df.to_csv(mode_export_dir / "cascade_partition_proteins_grouped.csv", index=False)
        manifest_df.to_csv(mode_export_dir / "cascade_partition_proteins_files_manifest.csv", index=False)

        print(f"[{mode}] Saved protein partition exports to: {mode_export_dir}")
else:
    print("SAVE_CASCADE_PROTEIN_EXPORTS is False; protein-list export skipped.")

## 15. What was removed from the merged version

The consolidated notebook intentionally removes repeated imports, repeated `utils_ccle.py` loading, repeated data/model/H5-split loading, repeated best-table construction, repeated metric-specific cascade code, repeated scTranslator loading/rescue helpers, and repeated export blocks.

It also removes the unused Spearman matrix load, the unused random split settings, and the empty proxy-correlation-threshold section whose `proxy_corr_assignments` table was never created in any of the three merged copies.